# PTB-XL ECG forecasting: standalone notebook

**Upload just this notebook** to Colab or open it in Jupyter or VS Code.
The Python functions, models, and configuration are included below. You do not
need a project ZIP, GitHub checkout, source directory, or configuration file.

Run the settings cell, then **Check the dataset**. In Colab the default folder is
**My Drive → Colab Notebooks → PTB-XL**. When the check passes, continue with the
rest of the notebook in pilot mode. Locally, set `LOCAL_DATA_ROOT` to your data
folder. Change patient counts in the settings cell. Fold 10 stays locked.

In [ ]:
import os
from pathlib import Path

# Start with pilot. Set EPOCHS=120 for the full study.
EXECUTION_ENV = 'auto'              # auto | local | colab
RUN_MODE = 'pilot'                 # pilot | medium | full
EPOCHS = 60                        # medium epochs; full requires 120
PILOT_EPOCHS = 2
BATCH_SIZE = 64
STUDY_ID = 'ptbxl_standalone_7s3s_v1'
RESUME = False
FINAL_TEST = False
TRAINING_DEVICE = 'auto'           # auto | cpu | cuda
ALLOW_FULL_CPU = False
ALLOW_PRIOR_HISTORY = False
TUNE_HIDDEN_128 = False

PILOT_TRAIN_PATIENTS = 300
PILOT_VAL_PATIENTS = 75
MEDIUM_TRAIN_PATIENTS = 1500
MEDIUM_VAL_PATIENTS = None         # None takes all of official fold 9

DATA_SOURCE = 'auto'               # auto: Drive in Colab, local folder elsewhere
SAVE_TO_DRIVE = False              # True keeps Colab results after a runtime reset

# Paths: edit the folders here. The rest of the notebook uses these variables.
# Local paths (relative to the notebook kernel's working directory unless absolute).
LOCAL_DATA_ROOT = 'PTB-XL'          # Use e.g. r'D:\Datasets\PTB-XL' for an absolute path
LOCAL_CACHE_DIR = 'data/study_v3'
LOCAL_RESULTS_DIR = 'results'

# Colab runtime paths. Change an individual folder here if needed.
COLAB_ROOT = Path('/content')
COLAB_DOWNLOAD_DIR = COLAB_ROOT / 'PTB-XL'
COLAB_CACHE_DIR = COLAB_ROOT / 'ptbxl_cache'
COLAB_RESULTS_DIR = COLAB_ROOT / 'ptbxl_research'

# Google Drive paths. The dataset and results are separate folders.
COLAB_DRIVE_MOUNT = COLAB_ROOT / 'drive'
MY_DRIVE_ROOT = COLAB_DRIVE_MOUNT / 'MyDrive'
DRIVE_DATA_ROOT = MY_DRIVE_ROOT / 'Colab Notebooks' / 'PTB-XL'
DRIVE_OUTPUT = MY_DRIVE_ROOT / 'ptbxl_research'

INSTALL_DEPENDENCIES = None        # None: install on Colab only; True: also install locally

SINGLE_RUN = False
ARCHITECTURE = 'lstm'              # lstm | cnn_lstm, used when SINGLE_RUN is True
FILTERING = 'P0'                   # P0 | P1
FEATURE_SET = 'F0'                 # F0 | F1 | F2 | expanded

VALIDATION_FIXTURE = os.environ.get('PTBXL_NOTEBOOK_FIXTURE')
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')

if EXECUTION_ENV not in ('auto', 'local', 'colab'):
    raise ValueError('EXECUTION_ENV must be auto, local, or colab.')
if RUN_MODE not in ('pilot', 'medium', 'full'):
    raise ValueError('RUN_MODE must be pilot, medium, or full.')
if TRAINING_DEVICE not in ('auto', 'cpu', 'cuda'):
    raise ValueError('TRAINING_DEVICE must be auto, cpu, or cuda.')
if RUN_MODE == 'full' and EPOCHS != 120:
    raise ValueError('Full mode requires EPOCHS=120.')
if RUN_MODE == 'full' and SINGLE_RUN:
    raise ValueError('Use pilot or medium for a manually chosen single run.')

## 0. Check the dataset before training

Run this cell immediately after the settings. In Colab, allow Drive access.
The **Loaded** column checks the required files, reads the ECG metadata and
checks one training recording's files. It does not start training or open
fold-10 waveforms. Section 3 can download the data if you explicitly choose
`DATA_SOURCE='physionet'`.

In [ ]:
import csv
from importlib.util import find_spec
from IPython.display import Markdown, display

runtime = EXECUTION_ENV
if VALIDATION_FIXTURE:
    runtime = 'local'
elif runtime == 'auto':
    try:
        running_in_colab = find_spec('google.colab') is not None
    except ModuleNotFoundError:
        running_in_colab = False
    runtime = 'colab' if running_in_colab else 'local'

required_dataset_files = ('ptbxl_database.csv', 'scp_statements.csv', 'SHA256SUMS.txt')
dataset_root = None
data_source = None
if VALIDATION_FIXTURE:
    print('Synthetic validation: the fixture is created in Section 3.')
else:
    data_source = DATA_SOURCE if DATA_SOURCE != 'auto' else ('drive' if runtime == 'colab' else 'local')
    if (data_source == 'drive' and runtime != 'colab') or (data_source == 'local' and runtime != 'local'):
        raise ValueError(f'DATA_SOURCE={data_source!r} is not available in {runtime} mode.')
    if data_source not in ('drive', 'local', 'physionet'):
        raise ValueError("Choose DATA_SOURCE='auto', 'drive', 'local', or 'physionet'.")
    if runtime == 'colab' and (data_source == 'drive' or SAVE_TO_DRIVE):
        from google.colab import drive
        drive.mount(str(Path(COLAB_DRIVE_MOUNT).expanduser()))

    if data_source == 'drive':
        dataset_root = Path(DRIVE_DATA_ROOT).expanduser().resolve()
    else:
        dataset_root = Path(COLAB_DOWNLOAD_DIR if runtime == 'colab' else LOCAL_DATA_ROOT).expanduser()
        if data_source == 'local' and not dataset_root.is_absolute():
            here = Path.cwd().resolve()
            dataset_root = next(
                (p / dataset_root for p in (here, *here.parents)
                 if (p / dataset_root / 'ptbxl_database.csv').is_file()),
                dataset_root,
            )
        dataset_root = dataset_root.resolve()

    print('Dataset folder:', dataset_root)
    if data_source == 'physionet':
        print('Official download selected. The files will be downloaded and validated in Section 3.')
    else:
        checks = [
            (name, (dataset_root / name).is_file(), 'Required dataset file')
            for name in required_dataset_files
        ]
        checks.append(('records100/', (dataset_root / 'records100').is_dir(), '100 Hz waveform directory'))

        if all(ok for _, ok, _ in checks):
            try:
                with (dataset_root / 'ptbxl_database.csv').open(encoding='utf-8-sig', newline='') as stream:
                    reader = csv.DictReader(stream)
                    needed = {'ecg_id', 'patient_id', 'strat_fold', 'filename_lr'}
                    if not needed.issubset(reader.fieldnames or ()):
                        raise ValueError('Missing required columns in ptbxl_database.csv')
                    records = 0
                    sample = None
                    for row in reader:
                        records += 1
                        if sample is None and row['strat_fold'] in ('1','2','3','4','5','6','7','8'):
                            sample = row
                if sample is None:
                    raise ValueError('No training record found in official folds 1–8')
                checks.append(('ECG metadata', True, f'{records:,} rows read'))
                relative = Path(sample['filename_lr'])
                if relative.is_absolute() or '..' in relative.parts or relative.parts[:1] != ('records100',):
                    raise ValueError('The sample filename does not point inside records100/')
                for extension in ('.hea', '.dat'):
                    with (dataset_root / relative.with_suffix(extension)).open('rb') as stream:
                        if not stream.read(1):
                            raise ValueError(f'The training sample {extension} file is empty')
                checks.append(('Training sample .hea and .dat', True, f'Both files readable (ECG {int(sample["ecg_id"])})'))
            except (OSError, ValueError, csv.Error) as error:
                checks.append(('Metadata or training sample', False, str(error).replace('|', '/').replace('\n', ' ')))

        rows = ['| File or check | Loaded | Details |', '| --- | --- | --- |']
        rows.extend(f"| {name} | {'Yes' if ok else 'No'} | {detail} |" for name, ok, detail in checks)
        display(Markdown('\n'.join(rows)))
        if not all(ok for _, ok, _ in checks):
            raise FileNotFoundError(
                f'Dataset check failed under {dataset_root}. Check the rows above. '
                'Use the folder that directly contains the CSV files and records100/.'
            )
        print('Dataset check passed. Continue with Section 1.')

## 1. Set up the runtime

In Colab, a GPU is optional for the pilot and required for full mode unless
explicitly overridden. Set `SAVE_TO_DRIVE=True` before a long Colab run. With
Drive saving disabled, use the Colab Files pane to download the results before
the runtime closes. A local run uses the selected Jupyter kernel.

In [ ]:
import hashlib
import subprocess
import sys

if 'runtime' not in globals():
    raise RuntimeError('Run the settings and dataset check cells first.')

dependencies = ['torch>=2.1,<3', 'numpy>=1.24,<3', 'pandas>=2.0,<4', 'matplotlib>=3.7,<4', 'wfdb>=4.1,<5', 'scipy>=1.11,<2', 'PyYAML>=6,<7']
install = runtime == 'colab' if INSTALL_DEPENDENCIES is None else INSTALL_DEPENDENCIES
if install and not VALIDATION_FIXTURE:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *dependencies], check=True)
try:
    import matplotlib
    import numpy as np
    import pandas as pd
    import scipy
    import torch
    import wfdb
    import yaml
except ModuleNotFoundError as error:
    raise RuntimeError(
        f'Missing notebook dependency: {error.name}. Set INSTALL_DEPENDENCIES=True or '
        'select a kernel with the required libraries installed.'
    ) from error

cuda_available = torch.cuda.is_available()
print(f'Environment: {runtime}; PyTorch: {torch.__version__}; CUDA: {cuda_available}')
if cuda_available:
    properties = torch.cuda.get_device_properties(0)
    print(f'GPU: {properties.name}; VRAM: {properties.total_memory / 2**30:.2f} GiB')
if not VALIDATION_FIXTURE:
    if TRAINING_DEVICE == 'cuda' and not cuda_available:
        raise RuntimeError('CUDA was requested but is unavailable in this kernel.')
    if RUN_MODE == 'full' and (not cuda_available or TRAINING_DEVICE == 'cpu') and not ALLOW_FULL_CPU:
        raise RuntimeError('Select a CUDA-enabled kernel or explicitly enable ALLOW_FULL_CPU.')

# Capture the actual implementation cells for checkpoint and result provenance.
implementation_cells = {}

def remember_source(name):
    history = get_ipython().history_manager.input_hist_raw
    source = history[-1].replace('\r\n', '\n').strip() + '\n'
    if not source.endswith(f'remember_source("{name}")\n'):
        raise RuntimeError(f'Cannot identify the executed implementation cell: {name}')
    implementation_cells[name] = source

## 2. Python implementation (already included)

These cells define the same study components without importing any project
module. Run them before the experiment cells. This notebook records the source
of these cells with its checkpoints. Use a separate study ID from the CLI.

### Configuration, checksums and device selection

In [ ]:
"""Configuration, atomic metadata writes, and scientific protocol identities."""

import copy
import hashlib
import json
from pathlib import Path

import yaml

SCHEMA = 3
LEADS = ["I", "II", "III", "aVR", "aVL", "aVF", "V1", "V2", "V3", "V4", "V5", "V6"]
CLASSES = ["NORM", "MI", "STTC", "CD", "HYP"]
FEATURE_SETS = ("F0", "F1", "F2", "expanded", "mrmr", "permutation")


def digest(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, allow_nan=False).encode()).hexdigest()


def file_hash(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(value, indent=2, allow_nan=False), encoding="utf-8")
    temporary.replace(path)


def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8-sig"))


DEFAULT_CONFIG_YAML = """
schema: 3
study_id: primary_7s3s_v3
run_mode: pilot
data:
  root: PTB-XL
  sampling_rate: 100
  history: 700
  horizon: 300
  synthetic: false
cohorts:
  pilot: {train: 300, val: 75}
  medium: {train: 1500, val: null}
preprocessing:
  order: 4
  low_hz: 0.5
  high_hz: 40.0
features:
  prior_history: false
  selected_groups: 3
  permutation_repeats: 5
model:
  hidden: 64
  depth: 1
  dropout: 0.2
  head_width: 128
  static_embedding: 16
training:
  seed: 42
  epochs: 60
  pilot_epochs: 2
  full_epochs: 120
  batch_size: 64
  lr: 0.0005
  learning_rates: [0.0001, 0.0005, 0.001]
  tune_hidden_128: false
  weight_decay: 0.0001
  gradient_clip: 1.0
  num_workers: 0
  threads: 4
  device: auto
  allow_full_cpu: false
evaluation:
  bootstrap_samples: 1000
  final_test: false
  example_ecg_id: null
paths:
  cache: data/study_v3
  results: results
"""


def load_config(mode=None):
    cfg = yaml.safe_load(DEFAULT_CONFIG_YAML)
    if mode:
        cfg["run_mode"] = mode
    validate(cfg)
    return cfg


def validate(cfg):
    d, tr = cfg["data"], cfg["training"]
    if cfg["run_mode"] not in ("pilot", "medium", "full"):
        raise ValueError("run_mode must be pilot, medium or full")
    if d["sampling_rate"] != 100 or (d["history"], d["horizon"]) not in ((700, 300), (500, 500)):
        raise ValueError("Use native 100 Hz and either 700/300 or the separate 500/500 ablation")
    if tr["epochs"] < 1 or tr["batch_size"] < 1 or tr["lr"] <= 0:
        raise ValueError("Invalid training budget, batch size or learning rate")
    if cfg["model"]["depth"] not in (1, 2):
        raise ValueError("Use one or two recurrent layers")
    p = cfg["preprocessing"]
    if not 0 < p["low_hz"] < p["high_hz"] < 50 or p["order"] < 1:
        raise ValueError("Invalid bandpass settings")


def protocol(cfg):
    return {
        "schema": SCHEMA,
        "dataset": "PTB-XL",
        "version": "1.0.3",
        "fs": 100,
        "history": cfg["data"]["history"],
        "horizon": cfg["data"]["horizon"],
        "start": 0,
        "windows_per_record": 1,
        "leads": LEADS,
        "target": "original unfiltered mV",
        "scaler": "raw training history per-lead zscore",
        "folds": {"train": list(range(1, 9)), "val": [9], "test": [10]},
    }


def cohort_limits(cfg):
    mode = cfg["run_mode"]
    return cfg["cohorts"][mode] if mode != "full" else {"train": None, "val": None}


def study_dir(cfg):
    return Path(cfg["paths"]["results"]) / cfg["study_id"]


def source_hashes():
    return {
        name: hashlib.sha256(source.encode("utf-8")).hexdigest()
        for name, source in sorted(implementation_cells.items())
    }


def device_for(cfg):
    import torch

    requested = cfg["training"]["device"]
    if requested == "cuda" and not torch.cuda.is_available():
        raise RuntimeError("CUDA requested but unavailable; select a GPU runtime or device=cpu")
    device = "cuda" if requested == "auto" and torch.cuda.is_available() else requested
    device = "cpu" if device == "auto" else device
    if cfg["run_mode"] == "full" and device != "cuda" and not cfg["training"]["allow_full_cpu"]:
        raise RuntimeError(
            "Full training requires CUDA; explicitly enable allow_full_cpu to override"
        )
    return torch.device(device)

remember_source("config.txt")

### Waveform preprocessing and optional static features

Every lead uses z = (x - mean) / std, with mean and std fitted only on raw training histories. P1 filters only the observed history before using the same scaler as P0.

In [ ]:
"""Causal history transforms and training-fitted waveform/static scaling."""

import ast
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import signal



def filter_history(history, cfg):
    """Accept [time, lead]; no future samples enter this function."""
    p = cfg["preprocessing"]
    sos = signal.butter(
        p["order"], [p["low_hz"], p["high_hz"]], fs=100, btype="bandpass", output="sos"
    )
    zi = signal.sosfilt_zi(sos)[:, :, None] * history[0][None, None, :]
    result, _ = signal.sosfilt(sos, history, axis=0, zi=zi)
    return result.astype(np.float32)


def waveform_scaler(waves, history, batch_size=128):
    total = np.zeros(12, dtype=np.float64)
    squares = total.copy()
    count = 0
    for start in range(0, len(waves), batch_size):
        x = np.asarray(waves[start : start + batch_size, :history], dtype=np.float64)
        total += x.sum(axis=(0, 1))
        squares += np.square(x).sum(axis=(0, 1))
        count += x.shape[0] * x.shape[1]
    mean = total / count
    std = np.sqrt(np.maximum(squares / count - mean**2, 0))
    return {
        "mean": mean.tolist(),
        "std": np.where(std > 1e-8, std, 1).tolist(),
        "fitted_split": "train",
        "fitted_samples_per_lead": count,
        "definition": "raw historical inputs only; population standard deviation",
    }


def inverse_scale(x, scaler):
    return np.asarray(x) * np.asarray(scaler["std"]) + np.asarray(scaler["mean"])


def prepare_transforms(cache, cfg, preprocessing, *, include_test=False):
    cache = Path(cache)
    base = read_json(cache / "manifest.json")
    history = cfg["data"]["history"]
    scaler_path = cache / "scaler.json"
    if not scaler_path.exists():
        waves = np.load(cache / "waves_train.npy", mmap_mode="r")
        write_json(scaler_path, {**waveform_scaler(waves, history), "cohort": base["cohort"]})
        write_json(cache / "scaler_receipt.json", {"sha256": file_hash(scaler_path)})
    if read_json(cache / "scaler_receipt.json")["sha256"] != file_hash(scaler_path):
        raise ValueError("Saved training scaler changed")
    scaler = read_json(scaler_path)
    if scaler.get("cohort") != base["cohort"]:
        raise ValueError("Scaler belongs to a different cohort")
    if preprocessing not in ("P0", "P1"):
        raise ValueError("preprocessing must be P0 or P1")
    if preprocessing == "P0":
        return scaler
    if include_test:

        verify_lock(cfg, cache)
    identity = digest({"cohort": base["cohort"], "settings": cfg["preprocessing"]})
    splits = ("train", "val", "test") if include_test else ("train", "val")
    for split in splits:
        raw_path = cache / f"waves_{split}.npy"
        if not raw_path.exists():
            continue
        path = cache / f"P1_{identity[:12]}_{split}.npy"
        receipt = path.with_suffix(".json")
        if path.exists() and receipt.exists():
            info = read_json(receipt)
            if info["identity"] != identity or info["sha256"] != file_hash(path):
                raise ValueError("Filtered cache changed; use a fresh cache directory")
            continue
        waves = np.load(raw_path, mmap_mode="r")
        output = np.lib.format.open_memmap(
            path, mode="w+", dtype="float32", shape=(len(waves), history, 12)
        )
        for i in range(len(waves)):
            output[i] = filter_history(waves[i, :history], cfg)
        output.flush()
        del output
        write_json(
            receipt,
            {
                "identity": identity,
                "sha256": file_hash(path),
                "target": "unchanged raw future",
                "scaler_sha256": file_hash(scaler_path),
            },
        )
    return scaler


def filtered_path(cache, cfg, split):
    base = read_json(Path(cache) / "manifest.json")
    key = digest({"cohort": base["cohort"], "settings": cfg["preprocessing"]})[:12]
    return Path(cache) / f"P1_{key}_{split}.npy"


def prior_diagnoses(frame, root):
    """Group equal timestamps before updating history, excluding self and same-time labels."""
    mapping = pd.read_csv(Path(root) / "scp_statements.csv", index_col=0)
    mapping = mapping.loc[mapping.diagnostic.eq(1), "diagnostic_class"].dropna().to_dict()
    labels = np.zeros((len(frame), 5), dtype=np.float32)
    for i, codes in enumerate(frame.scp_codes):
        parsed = ast.literal_eval(codes)
        if not isinstance(parsed, dict):
            raise ValueError("SCP annotations must be dictionaries")
        classes = {mapping.get(k) for k in parsed}
        labels[i] = [name in classes for name in CLASSES]
    prior = np.zeros((len(frame), 6), dtype=np.float32)
    prior[:, -1] = 1
    work = frame[["patient_id", "recording_date"]].copy()
    work["position"] = np.arange(len(frame))
    work["date"] = pd.to_datetime(work.recording_date, errors="coerce")
    edges = []
    for _, patient in work.groupby("patient_id", sort=False):
        known, previous = np.zeros(5, dtype=np.float32), []
        for date, same_time in patient.dropna(subset=["date"]).groupby("date", sort=True):
            indices = same_time.position.to_numpy(dtype=int)
            prior[indices, :5] = known
            prior[indices, -1] = float(not previous)
            for i in indices:
                for j, previous_date in previous:
                    if not previous_date < date:
                        raise ValueError("Non-past diagnosis entered historical features")
                    edges.append((int(frame.iloc[j].ecg_id), int(frame.iloc[i].ecg_id)))
            known = np.maximum(known, labels[indices].max(axis=0))
            previous.extend((i, date) for i in indices)
    return prior, {
        "passed": True,
        "strict_prior_edges": len(edges),
        "edge_hash": digest(edges),
        "available_records": int((prior[:, -1] == 0).sum()),
        "missing_dates": int(work.date.isna().sum()),
        "interpretation": "prior ECG annotation proxies, not a clinical problem list",
    }


def observed_rr(history):
    from wfdb import processing

    if np.ptp(history[:, 1]) < 1e-8:
        return [np.nan] * 3
    try:
        peaks = processing.xqrs_detect(history[:, 1], fs=100, verbose=False)
        rr = np.diff(peaks) / 100
        return [60 / rr.mean(), rr.mean(), rr.std()] if len(rr) >= 2 else [np.nan] * 3
    except (ValueError, IndexError, FloatingPointError):
        return [np.nan] * 3


def feature_arrays(cfg, cache, feature_set, splits=None):
    """Return train-fitted static arrays; expensive observed features are extension-only."""
    cache = Path(cache)
    splits = list(splits) if splits is not None else ["train", "val"]
    if "test" in splits:

        verify_lock(cfg, cache)
    frames = {s: pd.read_csv(cache / f"metadata_{s}.csv") for s in splits}
    full = pd.concat([frames[s].assign(split=s) for s in splits], ignore_index=True)
    train = full.split.eq("train").to_numpy()
    base = read_json(cache / "manifest.json")
    if feature_set == "F0":
        return (
            {s: np.zeros((len(frames[s]), 0), dtype=np.float32) for s in splits},
            [],
            {"groups": {}, "statistics": {}},
        )
    raw, names, groups, stats = [], [], {}, {}

    def continuous(name, values):
        values = np.asarray(values, dtype=float)
        observed = values[train & np.isfinite(values)]
        center = float(np.median(observed)) if len(observed) else 0.0
        std = float(observed.std()) if len(observed) and observed.std() > 1e-8 else 1.0
        missing = ~np.isfinite(values)
        raw.extend([(np.where(missing, center, values) - center) / std, missing])
        names.extend([name, name + "_missing"])
        groups[name] = [name, name + "_missing"]
        stats[name] = {"median": center, "std": std, "training_observations": len(observed)}

    age = pd.to_numeric(full.age, errors="coerce").where(lambda a: a.between(0, 120))
    continuous("age", age)
    sex = pd.to_numeric(full.sex, errors="coerce")
    valid = sex.isin([0, 1])
    observed_sex = sex[train & valid]
    fill = float(observed_sex.mode().iloc[0]) if len(observed_sex) else 0.0
    raw.extend([sex.where(valid, fill).to_numpy(), (~valid).to_numpy()])
    names.extend(["sex", "sex_missing"])
    groups["sex"] = ["sex", "sex_missing"]
    stats["sex"] = {"training_mode": fill}
    prior_audit = None
    if feature_set == "F2" or (feature_set == "expanded" and cfg["features"]["prior_history"]):
        if not cfg["features"]["prior_history"]:
            raise ValueError("Enable and audit prior_history explicitly before F2")
        prior, prior_audit = prior_diagnoses(full, cfg["data"]["root"])
        prior_names = ["prior_" + c for c in CLASSES] + ["history_missing"]
        raw.extend(prior.T)
        names.extend(prior_names)
        groups["history"] = prior_names
    if feature_set == "expanded":
        height = pd.to_numeric(
            full.get("height", pd.Series(np.nan, index=full.index)), errors="coerce"
        )
        weight = pd.to_numeric(
            full.get("weight", pd.Series(np.nan, index=full.index)), errors="coerce"
        )
        height, weight = (
            height.where(height.between(50, 250)),
            weight.where(weight.between(10, 400)),
        )
        for name, values in (
            ("height", height),
            ("weight", weight),
            ("bmi", weight / (height / 100) ** 2),
        ):
            continuous(name, values)
        rr_key = digest(
            {"cohort": base["cohort"], "splits": splits, "history": cfg["data"]["history"]}
        )
        rr_path = cache / f"observed_rr_{rr_key[:12]}.npy"
        receipt = rr_path.with_suffix(".json")
        if receipt.exists() and read_json(receipt).get("identity") == rr_key:
            if file_hash(rr_path) != read_json(receipt)["sha256"]:
                raise ValueError("Observed RR cache changed")
            rr = np.load(rr_path)
        else:
            values = []
            for split in splits:
                waves = np.load(cache / f"waves_{split}.npy", mmap_mode="r")
                values.extend(observed_rr(w[: cfg["data"]["history"]]) for w in waves)
            rr = np.asarray(values)
            np.save(rr_path, rr)
            write_json(receipt, {"identity": rr_key, "sha256": file_hash(rr_path)})
        for i, name in enumerate(("heart_rate", "rr_mean", "rr_std")):
            continuous(name, rr[:, i])
    matrix = np.column_stack(raw).astype(np.float32)
    arrays = {s: matrix[full.split.eq(s)].copy() for s in splits}
    return (
        arrays,
        names,
        {
            "groups": groups,
            "statistics": stats,
            "prior_audit": prior_audit,
            "fitted_split": "train",
            "cohort": base["cohort"],
        },
    )

remember_source("transforms.txt")

### Official dataset, caches and loaders

In [ ]:
"""Validate PTB-XL, preserve official patient folds, and serve memory-mapped records."""

import urllib.request
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import wfdb
from torch.utils.data import DataLoader, Dataset


OFFICIAL_METADATA = {
    "ptbxl_database.csv": "7600de9c1b27d181d850b3c6038a35d7c3ddb6bb33b702e3a20252a6859d216b",
    "scp_statements.csv": "ad05b0b1fcae83bb1230755ad9cfc7c96f303feddc08a4a9ad5bdc9ca63bac8f",
}


def metadata(root):
    frame = (
        pd.read_csv(Path(root) / "ptbxl_database.csv").sort_values("ecg_id").reset_index(drop=True)
    )
    required = {"ecg_id", "patient_id", "strat_fold", "filename_lr", "age", "sex"}
    if required - set(frame) or frame.ecg_id.duplicated().any():
        raise ValueError("Missing metadata columns or duplicate ECG IDs")
    if (
        frame[["patient_id", "strat_fold"]].isna().any().any()
        or not frame.strat_fold.isin(range(1, 11)).all()
    ):
        raise ValueError("Invalid patient/fold identifiers")
    if frame.groupby("patient_id").strat_fold.nunique().max() != 1:
        raise ValueError("Patient appears in different official folds")
    frame["split"] = np.where(
        frame.strat_fold <= 8, "train", np.where(frame.strat_fold == 9, "val", "test")
    )
    for value in frame.filename_lr:
        p = Path(str(value))
        if p.is_absolute() or ".." in p.parts or p.parts[0] != "records100":
            raise ValueError("filename_lr must be a relative records100 path")
    return frame


def select_patients(frame, cfg):
    selected = {}
    limits = cohort_limits(cfg)
    for index, split in enumerate(("train", "val")):
        ids = np.sort(frame.loc[frame.split.eq(split), "patient_id"].unique())
        count = limits[split]
        if count is not None:
            if not 1 <= count <= len(ids):
                raise ValueError(f"Requested {count} {split} patients but only {len(ids)} exist")
            rng = np.random.default_rng(np.random.SeedSequence([cfg["training"]["seed"], index]))
            ids = np.sort(rng.permutation(ids)[:count])
        selected[split] = ids.tolist()
    selected["test"] = sorted(frame.loc[frame.split.eq("test"), "patient_id"].unique().tolist())
    return selected


def source_checksums(root):
    path = Path(root) / "SHA256SUMS.txt"
    if not path.exists():
        raise ValueError("Missing official SHA256SUMS.txt; retain it with the v1.0.3 dataset")
    values = {}
    for line in path.read_text().splitlines():
        checksum, name = line.split(maxsplit=1)
        values[name.strip().lstrip("*./")] = checksum
    return values


def verify_source(root, name, checksums):
    relative = Path(name).as_posix()
    path = Path(root) / relative
    if relative not in checksums or not path.is_file():
        raise ValueError(f"Missing source/checksum: {relative}")
    if file_hash(path) != checksums[relative]:
        raise ValueError(f"Source checksum mismatch: {relative}")


def read_wave(row, root):
    wave, info = wfdb.rdsamp(str(Path(root) / row.filename_lr))
    if wave.shape != (1000, 12) or info["fs"] != 100:
        raise ValueError(f"Unexpected waveform shape/rate for ECG {row.ecg_id}")
    if [n.upper() for n in info["sig_name"]] != [n.upper() for n in LEADS]:
        raise ValueError("Unexpected lead order")
    if info["units"] != ["mV"] * 12 or not np.isfinite(wave).all():
        raise ValueError("Expected finite physical mV waveforms")
    return wave.astype(np.float32)


def validate_cache(cache, cfg):
    cache = Path(cache)
    manifest = read_json(cache / "manifest.json")
    if manifest["schema"] != SCHEMA or manifest["protocol"] != protocol(cfg):
        raise ValueError("Cache protocol mismatch; legacy caches cannot be reused")
    unsigned = {k: v for k, v in manifest.items() if k != "fingerprint"}
    if digest(unsigned) != manifest["fingerprint"]:
        raise ValueError("Manifest fingerprint changed")
    root = Path(cfg["data"]["root"])
    for filename, checksum in manifest["metadata_hashes"].items():
        if not (root / filename).exists() or file_hash(root / filename) != checksum:
            raise ValueError("Source metadata changed or is unavailable")
    for filename, checksum in manifest["files"].items():
        if not (cache / filename).exists() or file_hash(cache / filename) != checksum:
            raise ValueError(f"Cache file changed: {filename}")
    for filename, checksum in manifest.get("waveform_source_hashes", {}).items():
        if not (root / filename).exists() or file_hash(root / filename) != checksum:
            raise ValueError(f"Source waveform changed: {filename}")
    for split in ("train", "val"):
        waves = np.load(cache / f"waves_{split}.npy", mmap_mode="r")
        if waves.shape != (manifest["splits"][split]["records"], 1000, 12):
            raise ValueError("Cached waveform shape mismatch")
    return manifest


def prepare(cfg, include_test=False, lock=None):
    root = Path(cfg["data"]["root"])
    frame = metadata(root)
    checksums = source_checksums(root)
    for name in ("ptbxl_database.csv", "scp_statements.csv"):
        verify_source(root, name, checksums)
    selected = select_patients(frame, cfg)
    metadata_hashes = {
        n: file_hash(root / n)
        for n in ("ptbxl_database.csv", "scp_statements.csv", "SHA256SUMS.txt")
    }
    if not cfg["data"].get("synthetic", False):
        if any(metadata_hashes[name] != expected for name, expected in OFFICIAL_METADATA.items()):
            raise ValueError("Metadata does not match the official PTB-XL v1.0.3 release")
        if len(frame) != 21799 or frame.patient_id.nunique() != 18869:
            raise ValueError("Unexpected v1.0.3 metadata population")
    identity = {
        "protocol": protocol(cfg),
        "patients": selected,
        "metadata_hashes": metadata_hashes,
        "synthetic": cfg["data"].get("synthetic", False),
    }
    cohort = digest(identity)
    cache = Path(cfg["paths"]["cache"]) / cohort[:16]
    cache.mkdir(parents=True, exist_ok=True)
    manifest_path = cache / "manifest.json"
    exists = manifest_path.exists()
    if exists:
        manifest = validate_cache(cache, cfg)
    else:
        manifest = {
            "schema": SCHEMA,
            "cohort": cohort,
            "protocol": protocol(cfg),
            "metadata_hashes": metadata_hashes,
            "selected_patients": selected,
            "synthetic": cfg["data"].get("synthetic", False),
            "splits": {},
            "files": {},
        }
    if include_test and (
        cfg["run_mode"] != "full"
        or not cfg["evaluation"]["final_test"]
        or lock is None
        or lock.get("cohort") != cohort
    ):
        raise ValueError(
            "Fold 10 requires full mode, explicit final_test and a matching frozen selection"
        )
    if include_test:

        if verify_lock(cfg, cache) != lock:
            raise ValueError("The supplied selection is not the persisted frozen selection")
    splits = ("train", "val", "test") if include_test else ("train", "val")
    for split in splits:
        rows = frame.loc[
            frame.split.eq(split) & frame.patient_id.isin(selected[split])
        ].reset_index(drop=True)
        if split == "test":
            manifest_test = cache / "test_manifest.json"
            if manifest_test.exists():
                saved = read_json(manifest_test)
                if saved["cohort"] != cohort or saved["counts"]["records"] != len(rows):
                    raise ValueError("Final test cache belongs to another cohort")
                for name, checksum in saved["files"].items():
                    if file_hash(cache / name) != checksum:
                        raise ValueError("Final test cache changed")
                for name, checksum in saved["sources"].items():
                    verify_source(root, name, {name: checksum})
                continue
        elif exists:
            continue
        path = cache / f"waves_{split}.npy"
        waves = np.lib.format.open_memmap(
            path, mode="w+", dtype="float32", shape=(len(rows), 1000, 12)
        )
        invalid, sources = [], {}
        for i, row in rows.iterrows():
            try:
                for suffix in (".hea", ".dat"):
                    name = str(row.filename_lr) + suffix
                    verify_source(root, name, checksums)
                    sources[name] = checksums[name]
                waves[i] = read_wave(row, root)
            except Exception as exc:
                invalid.append({"ecg_id": int(row.ecg_id), "reason": str(exc)})
        waves.flush()
        del waves
        if invalid:
            write_json(cache / f"invalid_{split}.json", invalid)
            raise ValueError(f"Invalid {split} records; see {cache}")
        rows.to_csv(cache / f"metadata_{split}.csv", index=False)
        files = {
            name: file_hash(cache / name)
            for name in (f"waves_{split}.npy", f"metadata_{split}.csv")
        }
        counts = {
            "records": len(rows),
            "patients": int(rows.patient_id.nunique()),
            "windows": len(rows),
        }
        if split == "test":
            write_json(
                cache / "test_manifest.json",
                {"cohort": cohort, "files": files, "sources": sources, "counts": counts},
            )
        else:
            manifest["files"].update(files)
            manifest["splits"][split] = counts
            manifest.setdefault("waveform_source_hashes", {}).update(sources)
    if not exists:
        manifest["dataset"] = {
            "records": len(frame),
            "patients": int(frame.patient_id.nunique()),
        }
        manifest["reserved_test"] = {
            "records": int(frame.split.eq("test").sum()),
            "patients": len(selected["test"]),
            "prepared": False,
        }
        manifest["fingerprint"] = digest(manifest)
        write_json(manifest_path, manifest)
    destination = study_dir(cfg) / cfg["run_mode"]
    previous = destination / "cohort.json"
    if previous.exists() and read_json(previous)["cohort"] != cohort:
        raise ValueError("This study/mode already has another cohort; use a new study_id")
    write_json(previous, manifest)
    selected_frame = frame.loc[frame.patient_id.isin(sum(selected.values(), []))].copy()
    columns = ["ecg_id", "patient_id", "strat_fold", "split", "filename_lr"]
    exported = selected_frame[columns].copy()
    exported["history_samples"] = cfg["data"]["history"]
    exported["future_samples"] = cfg["data"]["horizon"]
    exported["waveform_prepared"] = (
        ~exported.split.eq("test") | (cache / "test_manifest.json").exists()
    )
    exported.to_csv(destination / "cohort_manifest.csv", index=False)
    prepare_transforms(cache, cfg, "P0")
    return cache, manifest


class ECGDataset(Dataset):
    """Serve one complete-record forecast window without moving the cache onto GPU."""

    def __init__(self, cache, cfg, split, preprocessing, static):
        if split not in ("train", "val", "test"):
            raise ValueError("Unknown split")
        if split == "test":

            verify_lock(cfg, cache)
        self.waves = np.load(Path(cache) / f"waves_{split}.npy", mmap_mode="r")
        self.history, self.horizon = cfg["data"]["history"], cfg["data"]["horizon"]
        self.filtered = (
            np.load(filtered_path(cache, cfg, split), mmap_mode="r")
            if preprocessing == "P1"
            else None
        )
        scaler = read_json(Path(cache) / "scaler.json")
        self.mean = np.asarray(scaler["mean"], dtype=np.float32)
        self.std = np.asarray(scaler["std"], dtype=np.float32)
        self.static = np.asarray(static, dtype=np.float32)
        self.metadata = pd.read_csv(Path(cache) / f"metadata_{split}.csv")
        if self.static.shape[0] != len(self.waves):
            raise ValueError("Static feature/record count mismatch")

    def __len__(self):
        return len(self.waves)

    def __getitem__(self, i):
        raw = self.waves[i]
        x = self.filtered[i] if self.filtered is not None else raw[: self.history]
        y = raw[self.history : self.history + self.horizon]
        return {
            "x": torch.from_numpy(((x - self.mean) / self.std).copy()),
            "y": torch.from_numpy(((y - self.mean) / self.std).copy()),
            "static": torch.from_numpy(self.static[i].copy()),
            "patient_id": int(self.metadata.iloc[i].patient_id),
            "index": i,
        }


def loader(dataset, cfg, training=False, generator=None):
    return DataLoader(
        dataset,
        batch_size=cfg["training"]["batch_size"],
        shuffle=training,
        num_workers=cfg["training"]["num_workers"],
        generator=generator,
    )


def download(root, *, records=None):
    """Download only officially listed 100 Hz records and v1.0.3 metadata."""
    root = Path(root)
    root.mkdir(parents=True, exist_ok=True)
    base = "https://physionet.org/files/ptb-xl/1.0.3/"

    def fetch(name, expected=None):
        out = root / name
        if out.is_file() and (expected is None or file_hash(out) == expected):
            return
        out.parent.mkdir(parents=True, exist_ok=True)
        temporary = out.with_suffix(out.suffix + ".partial")
        with (
            urllib.request.urlopen(base + name, timeout=60) as response,
            temporary.open("wb") as stream,
        ):
            while block := response.read(1 << 16):
                stream.write(block)
        if expected is not None and file_hash(temporary) != expected:
            raise ValueError(f"Downloaded checksum mismatch: {name}")
        temporary.replace(out)

    for name in (
        "SHA256SUMS.txt",
        "ptbxl_database.csv",
        "scp_statements.csv",
        "LICENSE.txt",
        "ptbxl_v103_changelog.txt",
    ):
        fetch(name, OFFICIAL_METADATA.get(name))
    checksums = source_checksums(root)
    for name in ("ptbxl_database.csv", "scp_statements.csv"):
        verify_source(root, name, checksums)
        if checksums[name] != OFFICIAL_METADATA[name]:
            raise ValueError("Download metadata does not match the official v1.0.3 release")
    # Use pinned file URLs; wfdb.dl_database can prepend a second version.
    listed = metadata(root).filename_lr.tolist()
    selected = set(listed if records is None else records)
    if not selected.issubset(listed):
        raise ValueError("Downloads must be listed in official filename_lr metadata")
    files = [record + ext for record in listed if record in selected for ext in (".hea", ".dat")]
    with ThreadPoolExecutor(max_workers=8) as pool:
        for _ in pool.map(lambda name: fetch(name, checksums[name]), files):
            pass
    return root

remember_source("data.txt")

### Independent LSTM and CNN-LSTM models

Both models map [batch, 700, 12] to [batch, 300, 12]. For static conditioning, the learned metadata embedding is repeated at each history step. The CNN-LSTM extracts local patterns and pools the observed history before the recurrent layer.

In [ ]:
"""Direct multi-lead LSTM forecasting with optional early static conditioning."""

import torch
from torch import nn


class LSTMForecaster(nn.Module):
    """Summarize observed history and predict all future samples together."""

    def __init__(
        self,
        static_dim=0,
        horizon=300,
        hidden=64,
        depth=1,
        dropout=0.2,
        static_embedding=16,
        head_width=128,
    ):
        super().__init__()
        self.horizon = horizon
        self.static = (
            nn.Sequential(nn.Linear(static_dim, static_embedding), nn.ReLU())
            if static_dim
            else None
        )
        channels = 12 + (static_embedding if static_dim else 0)
        self.lstm = nn.LSTM(
            channels,
            hidden,
            num_layers=depth,
            batch_first=True,
            dropout=dropout if depth > 1 else 0,
        )
        self.head = nn.Sequential(
            nn.Linear(hidden, head_width),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(head_width, horizon * 12),
        )

    def forward(self, history, static=None):
        if history.ndim != 3 or history.shape[-1] != 12:
            raise ValueError("Expected history [batch,time,12]")
        if self.static is not None:
            if static is None:
                raise ValueError("This model requires static inputs")
            embedding = self.static(static)[:, None, :].expand(-1, history.shape[1], -1)
            history = torch.cat((history, embedding), dim=-1)
        _, (hidden, _) = self.lstm(history)
        return self.head(hidden[-1]).reshape(-1, self.horizon, 12)

"""Temporal convolutions followed by an LSTM and a direct forecast head."""

import torch
from torch import nn


class CNNLSTMForecaster(nn.Module):
    """Extract local observed patterns before recurrent forecasting."""

    def __init__(
        self,
        static_dim=0,
        horizon=300,
        hidden=64,
        depth=1,
        dropout=0.2,
        static_embedding=16,
        head_width=128,
    ):
        super().__init__()
        self.horizon = horizon
        self.static = (
            nn.Sequential(nn.Linear(static_dim, static_embedding), nn.ReLU())
            if static_dim
            else None
        )
        channels = 12 + (static_embedding if static_dim else 0)
        self.convolutions = nn.Sequential(
            nn.Conv1d(channels, 32, kernel_size=7, padding=3),
            nn.ReLU(),
            nn.Conv1d(32, 64, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.AvgPool1d(2),
        )
        self.lstm = nn.LSTM(
            64, hidden, num_layers=depth, batch_first=True, dropout=dropout if depth > 1 else 0
        )
        self.head = nn.Sequential(
            nn.Linear(hidden, head_width),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(head_width, horizon * 12),
        )

    def forward(self, history, static=None):
        if history.ndim != 3 or history.shape[-1] != 12:
            raise ValueError("Expected history [batch,time,12]")
        if self.static is not None:
            if static is None:
                raise ValueError("This model requires static inputs")
            embedding = self.static(static)[:, None, :].expand(-1, history.shape[1], -1)
            history = torch.cat((history, embedding), dim=-1)
        features = self.convolutions(history.transpose(1, 2)).transpose(1, 2)
        _, (hidden, _) = self.lstm(features)
        return self.head(hidden[-1]).reshape(-1, self.horizon, 12)

"""Two independent forecasting architectures; no contrastive projection head."""



def build_model(architecture, static_dim=0, horizon=300, **settings):
    if architecture == "lstm":
        return LSTMForecaster(static_dim=static_dim, horizon=horizon, **settings)
    if architecture == "cnn_lstm":
        return CNNLSTMForecaster(static_dim=static_dim, horizon=horizon, **settings)
    raise ValueError("architecture must be lstm or cnn_lstm")

remember_source("models.txt")

### Shared training, checkpointing and resume

The epoch function trains on normalized MSE: average((prediction - target)^2). Checkpoint selection uses validation patient-macro MAE in millivolts.

In [ ]:
"""One training loop, validation checkpoint selection, and exact-state resume."""

import copy
import os
import platform
import random
import time
from importlib.metadata import version
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.nn import functional as F



def seed_everything(seed, threads=4):
    os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.set_num_threads(threads)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.use_deterministic_algorithms(True, warn_only=True)


def rng_state(generator):
    return {
        "python": random.getstate(),
        "numpy": np.random.get_state(),
        "torch": torch.get_rng_state(),
        "loader": generator.get_state(),
        "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
    }


def restore_rng(state, generator):
    random.setstate(state["python"])
    np.random.set_state(state["numpy"])
    torch.set_rng_state(state["torch"].cpu())
    generator.set_state(state["loader"].cpu())
    if state["cuda"] is not None and torch.cuda.is_available():
        torch.cuda.set_rng_state_all([s.cpu() for s in state["cuda"]])


def save_checkpoint(path, state):
    path = Path(path)
    temporary = path.with_suffix(".tmp")
    torch.save(state, temporary)
    temporary.replace(path)


def static_inputs(cfg, cache, feature_set, selection=None, include_test=False):
    base = "expanded" if feature_set in ("mrmr", "permutation") else feature_set
    arrays, names, details = feature_arrays(
        cfg, cache, base, splits=("train", "val", "test") if include_test else ("train", "val")
    )
    if selection is not None:
        if selection["cohort"] != read_json(Path(cache) / "manifest.json")["cohort"]:
            raise ValueError("Feature selection belongs to another cohort")
        indices = [names.index(name) for name in selection["selected_features"]]
        arrays = {split: a[:, indices] for split, a in arrays.items()}
        names = selection["selected_features"]
        details["selection"] = selection
    elif feature_set in ("mrmr", "permutation"):
        raise ValueError("Selected-feature runs require a provenance-checked selection")
    return arrays, names, details


def epoch(model, batches, device, std, optimizer=None, gradient_clip=1.0):
    training = optimizer is not None
    model.train(training)
    records, total, count = [], 0.0, 0
    scale = torch.as_tensor(std, dtype=torch.float32, device=device)
    for batch in batches:
        x, y, static = [batch[k].to(device) for k in ("x", "y", "static")]
        with torch.set_grad_enabled(training):
            prediction = model(x, static)
            loss = F.mse_loss(prediction, y)
            if not torch.isfinite(loss):
                raise ValueError("Nonfinite loss")
            if training:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), gradient_clip)
                optimizer.step()
        error = (prediction.detach() - y) * scale
        records.extend(
            zip(
                batch["patient_id"].tolist(),
                error.abs().mean((1, 2)).cpu().tolist(),
                error.square().mean((1, 2)).cpu().tolist(),
                strict=True,
            )
        )
        total += loss.item() * len(x)
        count += len(x)
    if not count:
        raise ValueError("Empty DataLoader")
    frame = pd.DataFrame(records, columns=["patient", "mae", "mse"])
    return {
        "mse_normalized": total / count,
        "mae_mV": float(frame.mae.mean()),
        "rmse_mV": float(np.sqrt(frame.mse.mean())),
        "patient_macro_mae_mV": float(frame.groupby("patient").mae.mean().mean()),
    }


def train_run(
    cfg,
    cache,
    architecture,
    preprocessing="P0",
    feature_set="F0",
    *,
    name=None,
    epochs=None,
    lr=None,
    hidden=None,
    resume=False,
    selection=None,
    stop_after=None,
):
    cfg = copy.deepcopy(cfg)
    if lr is not None:
        cfg["training"]["lr"] = lr
    if hidden is not None:
        cfg["model"]["hidden"] = hidden
    mode, tr = cfg["run_mode"], cfg["training"]
    budget = epochs or (
        tr["pilot_epochs"]
        if mode == "pilot"
        else (tr["full_epochs"] if mode == "full" else tr["epochs"])
    )
    scaler = prepare_transforms(cache, cfg, preprocessing)
    arrays, names, feature_info = static_inputs(cfg, cache, feature_set, selection)
    manifest = read_json(Path(cache) / "manifest.json")
    seed_everything(tr["seed"], tr["threads"])
    device = device_for(cfg)
    model = build_model(architecture, len(names), cfg["data"]["horizon"], **cfg["model"]).to(device)
    parameters = sum(p.numel() for p in model.parameters() if p.requires_grad)
    name = (
        name
        or f"{architecture}_{preprocessing}_{feature_set}_lr{tr['lr']:g}_h{cfg['model']['hidden']}_e{budget}"
    )
    run = study_dir(cfg) / mode / "checkpoints" / name
    run.mkdir(parents=True, exist_ok=True)
    scientific = {
        "schema": SCHEMA,
        "cohort": manifest["cohort"],
        "preprocessing": preprocessing,
        "filter": cfg["preprocessing"] if preprocessing == "P1" else None,
        "scaler_sha256": file_hash(Path(cache) / "scaler.json"),
        "architecture": architecture,
        "model": cfg["model"],
        "features": names,
        "feature_statistics": feature_info,
        "budget": budget,
        "training": {
            k: tr[k] for k in ("seed", "lr", "batch_size", "weight_decay", "gradient_clip")
        },
        "source": source_hashes(),
        "versions": {p: version(p) for p in ("torch", "numpy", "scipy", "pandas", "wfdb")},
        "synthetic": manifest["synthetic"],
    }
    signature = digest(scientific)
    recorded = (run / "run.json").exists()
    if recorded:
        previous = read_json(run / "run.json")
        if previous.get("schema") != SCHEMA:
            raise ValueError("Legacy checkpoint: use the archived workflow")
        if previous["signature"] != signature:
            raise ValueError("Run configuration/source changed; create a distinct experiment ID")
    if (run / "last.pt").exists() and not resume:
        raise ValueError("Existing run; choose resume explicitly or a new experiment ID")
    if resume and not (run / "last.pt").exists():
        raise ValueError("No last checkpoint to resume")
    generator = torch.Generator().manual_seed(tr["seed"])
    batches = {
        s: loader(ECGDataset(cache, cfg, s, preprocessing, arrays[s]), cfg, s == "train", generator)
        for s in ("train", "val")
    }
    optimizer = torch.optim.AdamW(model.parameters(), lr=tr["lr"], weight_decay=tr["weight_decay"])
    history, best, elapsed = [], (float("inf"), float("inf")), 0.0
    if resume:
        state = torch.load(run / "last.pt", map_location=device, weights_only=False)
        if state.get("schema") != SCHEMA or state.get("signature") != signature:
            raise ValueError(
                "Checkpoint protocol is incompatible; use the notebook and settings that saved the run"
            )
        model.load_state_dict(state["model"])
        optimizer.load_state_dict(state["optimizer"])
        history, best, elapsed = state["history"], tuple(state["best"]), state["elapsed"]
        restore_rng(state["rng"], generator)
    record = {
        "schema": SCHEMA,
        "name": name,
        "signature": signature,
        "scientific": scientific,
        "config": cfg,
        "cache": str(Path(cache).resolve()),
        "architecture": architecture,
        "preprocessing": preprocessing,
        "feature_set": feature_set,
        "selected_features": names,
        "selection": selection,
        "parameters": parameters,
        "epoch_budget": budget,
        "smoke": mode == "pilot" or manifest["synthetic"],
        "environment": {
            "python": platform.python_version(),
            "torch": torch.__version__,
            "device": str(device),
            "gpu": torch.cuda.get_device_name() if device.type == "cuda" else None,
        },
    }
    if not recorded:
        write_json(run / "run.json", record)
        write_json(run / "cohort_manifest.json", manifest)
        write_json(run / "scaler.json", scaler)
    snapshot = study_dir(cfg) / "source" / digest(scientific["source"])[:16]
    for relative, expected in scientific["source"].items():
        target = snapshot / relative
        if target.exists() and file_hash(target) != expected:
            raise ValueError("Preserved source snapshot changed")
        if not target.exists():
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(implementation_cells[relative].encode("utf-8"))

    def persist_progress(state):
        number = state["epoch"]
        pd.DataFrame(state["history"]).to_csv(run / "history.csv", index=False)
        write_json(
            run / "status.json",
            {
                "state": "completed" if number == budget else "training incomplete",
                "epochs_completed": number,
                "budget": budget,
                "best_validation_mae_mV": state["best"][0],
                "runtime_seconds": state["elapsed"],
            },
        )

    if resume:
        # last.pt is the epoch transaction; repair derived files only if a write was interrupted.
        status = read_json(run / "status.json") if (run / "status.json").exists() else {}
        if status.get("epochs_completed") != len(history):
            save_checkpoint(run / "best.pt", state["best_checkpoint"])
            if len(history) in (60, 120):
                milestone = {
                    k: state[k] for k in ("schema", "signature", "model", "epoch", "objective")
                }
                save_checkpoint(run / f"epoch_{len(history)}.pt", milestone)
                save_checkpoint(run / f"best_through_{len(history)}.pt", state["best_checkpoint"])
            persist_progress(state)
        best_checkpoint = state["best_checkpoint"]
    else:
        best_checkpoint = None
    started = time.perf_counter()
    for number in range(len(history) + 1, budget + 1):
        tick = time.perf_counter()
        training = epoch(
            model, batches["train"], device, scaler["std"], optimizer, tr["gradient_clip"]
        )
        validation = epoch(model, batches["val"], device, scaler["std"])
        row = {
            "epoch": number,
            "seconds": time.perf_counter() - tick,
            "lr": tr["lr"],
            **{"train_" + k: v for k, v in training.items()},
            **{"val_" + k: v for k, v in validation.items()},
        }
        history.append(row)
        objective = (validation["patient_macro_mae_mV"], validation["rmse_mV"])
        selected = {
            "schema": SCHEMA,
            "signature": signature,
            "model": model.state_dict(),
            "epoch": number,
            "objective": objective,
        }
        if objective < best:
            best = objective
            best_checkpoint = {
                **selected,
                "model": {k: v.detach().cpu().clone() for k, v in selected["model"].items()},
            }
        state = {
            **selected,
            "optimizer": optimizer.state_dict(),
            "scheduler": None,
            "history": history,
            "best": best,
            "best_checkpoint": best_checkpoint,
            "rng": rng_state(generator),
            "elapsed": elapsed + time.perf_counter() - started,
        }
        save_checkpoint(run / "last.pt", state)
        if best_checkpoint["epoch"] == number:
            save_checkpoint(run / "best.pt", best_checkpoint)
        if number in (60, 120):
            save_checkpoint(run / f"epoch_{number}.pt", selected)
            save_checkpoint(run / f"best_through_{number}.pt", best_checkpoint)
        persist_progress(state)
        print(
            f"{name}: epoch {number}/{budget}; validation patient MAE {objective[0]:.6f} mV",
            flush=True,
        )
        if stop_after is not None and number >= stop_after:
            break
    return run

remember_source("training.txt")

### Physical-unit metrics and patient bootstrap

With e = forecast - true voltage in mV, MAE = average(|e|) and RMSE = sqrt(average(e^2)). Patient-macro MAE averages record errors within each patient before averaging across patients.

In [ ]:
"""Physical-mV errors and paired uncertainty with equal patient weighting."""

import numpy as np
import pandas as pd



def metric_summary(prediction, target, patients):
    p, y = np.asarray(prediction, dtype=np.float64), np.asarray(target, dtype=np.float64)
    if p.shape != y.shape or p.ndim != 3 or p.shape[-1] != 12 or not len(p):
        raise ValueError("Expected nonempty matching [record,time,12] arrays")
    if len(patients) != len(p) or not np.isfinite(p).all() or not np.isfinite(y).all():
        raise ValueError("Invalid patient count or nonfinite metric inputs")
    error = p - y
    per_record = pd.DataFrame(
        {
            "patient_id": patients,
            "mae_mV": np.abs(error).mean(axis=(1, 2)),
            "mse_mV2": np.square(error).mean(axis=(1, 2)),
        }
    )
    mse = float(np.square(error).mean())
    centered_p, centered_y = p - p.mean(axis=(0, 1)), y - y.mean(axis=(0, 1))
    denominator = np.sqrt(
        np.square(centered_p).sum(axis=(0, 1)) * np.square(centered_y).sum(axis=(0, 1))
    )
    correlation = np.divide(
        (centered_p * centered_y).sum(axis=(0, 1)),
        denominator,
        out=np.full(12, np.nan),
        where=denominator > 1e-12,
    )
    ss = float(np.square(y - y.mean()).sum())
    by_lead = pd.DataFrame(
        {
            "lead": LEADS,
            "mae_mV": np.abs(error).mean(axis=(0, 1)),
            "rmse_mV": np.sqrt(np.square(error).mean(axis=(0, 1))),
            "pearson": correlation,
        }
    )
    summary = {
        "mae_mV": float(per_record.mae_mV.mean()),
        "mse_mV2": mse,
        "rmse_mV": float(np.sqrt(mse)),
        "patient_macro_mae_mV": float(per_record.groupby("patient_id").mae_mV.mean().mean()),
        "r2": float(1 - np.square(error).sum() / ss) if ss > 1e-12 else None,
        "pearson_mean": float(np.nanmean(correlation)) if np.isfinite(correlation).any() else None,
        "records": len(p),
        "patients": int(per_record.patient_id.nunique()),
    }
    return summary, by_lead, per_record


def bootstrap_pair(first, second, samples=1000, seed=42):
    if set(first.ecg_id) != set(second.ecg_id):
        raise ValueError("Cannot compare different held-out record sets")
    joined = first[["ecg_id", "patient_id", "mae_mV"]].merge(
        second[["ecg_id", "patient_id", "mae_mV"]],
        on=["ecg_id", "patient_id"],
        validate="one_to_one",
        suffixes=("_a", "_b"),
    )
    if len(joined) != len(first):
        raise ValueError("Patient IDs differ between comparisons")
    joined["difference"] = joined.mae_mV_b - joined.mae_mV_a
    delta = joined.groupby("patient_id").difference.mean().to_numpy()
    rng = np.random.default_rng(seed)
    boot = [rng.choice(delta, size=len(delta), replace=True).mean() for _ in range(samples)]
    low, high = np.quantile(boot, [0.025, 0.975])
    return {
        "difference_B_minus_A_mV": float(delta.mean()),
        "ci95_low": float(low),
        "ci95_high": float(high),
        "patients": len(delta),
        "bootstrap_samples": samples,
        "conclusion": "helped" if high < 0 else ("did not help" if low > 0 else "inconclusive"),
    }

remember_source("metrics.txt")

### Inference, baselines and the fold-10 lock

In [ ]:
"""Explicit inference, raw-history baselines, and a frozen final-test ledger."""

import hashlib
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import torch



def outputs(cfg, name):
    path = study_dir(cfg) / cfg["run_mode"] / "metrics" / name
    path.mkdir(parents=True, exist_ok=True)
    return path


def predict(model, batches, device):
    model.eval()
    predictions, targets = [], []
    with torch.inference_mode():
        for batch in batches:
            predictions.append(
                model(batch["x"].to(device), batch["static"].to(device)).cpu().numpy()
            )
            targets.append(batch["y"].numpy())
    return np.concatenate(predictions), np.concatenate(targets)


def verify_lock(cfg, cache):
    path = study_dir(cfg) / "full" / "selection.json"
    if cfg["run_mode"] != "full" or not cfg["evaluation"]["final_test"] or not path.exists():
        raise ValueError(
            "Fold 10 is locked: full mode, explicit final_test and a frozen selection are required"
        )
    lock = read_json(path)
    manifest = read_json(Path(cache) / "manifest.json")
    if (
        lock["schema"] != SCHEMA
        or lock["cohort"] != manifest["cohort"]
        or lock["protocol"] != protocol(cfg)
        or lock["manifest_sha256"] != file_hash(Path(cache) / "manifest.json")
    ):
        raise ValueError("Frozen protocol/cohort mismatch")
    if lock["scaler_sha256"] != file_hash(Path(cache) / "scaler.json"):
        raise ValueError("Frozen scaler changed")
    if lock["filter"] != cfg["preprocessing"]:
        raise ValueError("Frozen filter settings changed")
    for name, item in lock["runs"].items():
        run = study_dir(cfg) / "full" / "checkpoints" / name
        if item["checkpoint_sha256"] != file_hash(run / "best.pt") or item[
            "run_sha256"
        ] != file_hash(run / "run.json"):
            raise ValueError("A frozen checkpoint or configuration changed")
        if item["validation_sha256"] != file_hash(outputs(cfg, name) / "val.json"):
            raise ValueError("Frozen validation evidence changed")
        if item["validation_receipt_sha256"] != file_hash(outputs(cfg, name) / "val_receipt.json"):
            raise ValueError("Frozen validation receipt changed")
        if read_json(run / "run.json")["scientific"]["source"] != source_hashes():
            raise ValueError("Source differs from the frozen scientific implementation")
        verify_evaluation(outputs(cfg, name), "val")
    return lock


def save_evaluation(cfg, cache, name, split, prediction, target, details):
    out = outputs(cfg, name)
    metadata = pd.read_csv(Path(cache) / f"metadata_{split}.csv")
    # Choose the displayed example before scoring; the other example uses median error.
    specified = cfg["evaluation"].get("example_ecg_id")
    if specified is None:
        specified = int(metadata.ecg_id.min())
    if specified not in set(metadata.ecg_id):
        raise ValueError(f"Pre-specified ECG {specified} is not in the evaluated cohort")
    summary, leads, records = metric_summary(prediction, target, metadata.patient_id.to_numpy())
    records.insert(0, "ecg_id", metadata.ecg_id.to_numpy())
    manifest = read_json(Path(cache) / "manifest.json")
    summary.update(
        name=name,
        split=split,
        cohort=manifest["cohort"],
        protocol=protocol(cfg),
        synthetic=manifest["synthetic"],
        smoke=cfg["run_mode"] == "pilot" or manifest["synthetic"],
        **details,
    )
    summary["target_sha256"] = hashlib.sha256(
        np.ascontiguousarray(target, dtype=np.float32).tobytes()
    ).hexdigest()
    records.to_csv(out / f"{split}_records.csv", index=False)
    leads.to_csv(out / f"{split}_leads.csv", index=False)
    np.savez_compressed(
        out / f"{split}_predictions.npz",
        prediction=prediction,
        target=target,
        ecg_id=metadata.ecg_id.to_numpy(),
        patient_id=metadata.patient_id.to_numpy(),
    )
    summary["pre_specified_ecg_id"] = int(specified)
    order = records.sort_values(["mae_mV", "ecg_id"], kind="stable")
    summary["median_error_ecg_id"] = int(order.iloc[len(order) // 2].ecg_id)
    write_json(out / f"{split}.json", summary)
    files = {
        p.name: file_hash(p)
        for p in (
            out / f"{split}.json",
            out / f"{split}_records.csv",
            out / f"{split}_leads.csv",
            out / f"{split}_predictions.npz",
        )
    }
    write_json(out / f"{split}_receipt.json", {"files": files, "cohort": manifest["cohort"]})
    return summary


def verify_evaluation(out, split):
    out = Path(out)
    receipt = read_json(out / f"{split}_receipt.json")
    for name, expected in receipt["files"].items():
        if not (out / name).exists() or file_hash(out / name) != expected:
            raise ValueError(f"Saved evaluation artifact changed: {name}")
    return receipt


def evaluate_run(cfg, cache, run, split="val", checkpoint="best.pt", label=None):
    if split not in ("val", "test"):
        raise ValueError("Evaluation split must be val or test")
    run = Path(run)
    record = read_json(run / "run.json")
    if record.get("schema") != SCHEMA:
        raise ValueError("Legacy checkpoint: use the archived workflow")
    if record["scientific"]["source"] != source_hashes():
        raise ValueError("Use the recorded source snapshot to evaluate this checkpoint")
    if split == "test":
        lock = verify_lock(cfg, cache)
        if run.name not in lock["runs"] or checkpoint != "best.pt":
            raise ValueError("Only frozen selected checkpoints may evaluate fold 10")
    manifest = read_json(Path(cache) / "manifest.json")
    if record["scientific"]["cohort"] != manifest["cohort"]:
        raise ValueError("Evaluation cohort differs from training")
    if record["scientific"]["scaler_sha256"] != file_hash(Path(cache) / "scaler.json"):
        raise ValueError("Evaluation scaler differs from training")
    if record["preprocessing"] == "P1" and record["scientific"]["filter"] != cfg["preprocessing"]:
        raise ValueError("Evaluation filter differs from training")
    scaler = prepare_transforms(cache, cfg, record["preprocessing"], include_test=split == "test")
    arrays, names, _ = static_inputs(
        record["config"]
        | {
            "data": cfg["data"],
            "evaluation": cfg["evaluation"],
            "paths": cfg["paths"],
            "study_id": cfg["study_id"],
        },
        cache,
        record["feature_set"],
        record["selection"],
        include_test=split == "test",
    )
    if names != record["selected_features"]:
        raise ValueError("Evaluation feature order differs from training")
    dataset = ECGDataset(cache, cfg, split, record["preprocessing"], arrays[split])
    device = device_for(cfg)
    model = build_model(
        record["architecture"], len(names), cfg["data"]["horizon"], **record["config"]["model"]
    ).to(device)
    state = torch.load(run / checkpoint, map_location=device, weights_only=True)
    if state["schema"] != SCHEMA or state["signature"] != record["signature"]:
        raise ValueError("Checkpoint signature mismatch")
    model.load_state_dict(state["model"])
    prediction, target = predict(model, loader(dataset, cfg), device)
    prediction = inverse_scale(prediction, scaler)
    # Use the stored physical truth directly, avoiding normalization round-off in ground truth.
    target = np.asarray(dataset.waves[:, cfg["data"]["history"] :])
    result = save_evaluation(
        cfg,
        cache,
        label or run.name,
        split,
        prediction,
        target,
        {
            "architecture": record["architecture"],
            "preprocessing": record["preprocessing"],
            "feature_set": record["feature_set"],
            "parameters": record["parameters"],
            "best_epoch": state["epoch"],
            "checkpoint_sha256": file_hash(run / checkpoint),
            "run_signature": record["signature"],
            "runtime_seconds": read_json(run / "status.json")["runtime_seconds"],
        },
    )
    print(
        f"{run.name} {split}: patient-macro MAE={result['patient_macro_mae_mV']:.6f} mV", flush=True
    )
    return result


def baseline(cfg, cache, name, split="val"):
    if split not in ("val", "test"):
        raise ValueError("Baselines evaluate validation or locked final test only")
    if split == "test":
        verify_lock(cfg, cache)
    if name not in ("persistence", "repeat_window"):
        raise ValueError("Supported baselines: persistence, repeat_window; ARIMA is deferred")
    raw = np.load(Path(cache) / f"waves_{split}.npy", mmap_mode="r")
    h, f = cfg["data"]["history"], cfg["data"]["horizon"]
    target = np.asarray(raw[:, h : h + f])
    prediction = (
        np.repeat(raw[:, h - 1 : h], f, axis=1)
        if name == "persistence"
        else np.asarray(raw[:, h - f : h]).copy()
    )
    return save_evaluation(
        cfg,
        cache,
        name,
        split,
        prediction,
        target,
        {
            "architecture": "baseline",
            "preprocessing": "raw",
            "feature_set": "F0",
            "parameters": 0,
            "definition": "repeat last observed voltage"
            if name == "persistence"
            else "repeat last horizon-length observed segment",
        },
    )


def ranking_key(row):
    return (
        row["patient_macro_mae_mV"],
        row["rmse_mV"],
        row.get("parameters", 0),
        row.get("runtime_seconds", 0),
        row["name"],
    )


def freeze_selection(cfg, cache, runs):
    if cfg["run_mode"] != "full":
        raise ValueError("Only full-population validation may freeze final selection")
    manifest = read_json(Path(cache) / "manifest.json")
    path = study_dir(cfg) / "full" / "selection.json"
    items, recipes, highlights = {}, {}, {}
    for run in map(Path, runs):
        record, status = read_json(run / "run.json"), read_json(run / "status.json")
        expected = 120 if not manifest["synthetic"] else cfg["training"]["full_epochs"]
        if (
            record["scientific"]["cohort"] != manifest["cohort"]
            or status["epochs_completed"] != expected
            or record["epoch_budget"] != expected
        ):
            raise ValueError(
                "Final conditions must complete the full 120-epoch budget on this cohort"
            )
        metrics_path = outputs(cfg, run.name) / "val.json"
        verify_evaluation(metrics_path.parent, "val")
        metrics = read_json(metrics_path)
        if metrics["checkpoint_sha256"] != file_hash(run / "best.pt"):
            raise ValueError("Validation evaluation is stale")
        if record["scientific"]["scaler_sha256"] != file_hash(Path(cache) / "scaler.json"):
            raise ValueError("Validation scaler is stale")
        if (
            record["preprocessing"] == "P1"
            and record["scientific"]["filter"] != cfg["preprocessing"]
        ):
            raise ValueError("Validation filter is stale")
        pair = (record["architecture"], record["preprocessing"])
        if pair in recipes or record["feature_set"] not in ("F0", "F1"):
            raise ValueError(
                "Final selection requires one F0/F1 run for each model/filter condition"
            )
        recipes[pair] = record
        items[run.name] = {
            "checkpoint_sha256": file_hash(run / "best.pt"),
            "run_sha256": file_hash(run / "run.json"),
            "validation_sha256": file_hash(metrics_path),
            "validation_receipt_sha256": file_hash(metrics_path.with_name("val_receipt.json")),
            "recipe": {
                "architecture": record["architecture"],
                "preprocessing": record["preprocessing"],
                "feature_set": record["feature_set"],
                "lr": record["config"]["training"]["lr"],
                "hidden": record["config"]["model"]["hidden"],
                "selected_epoch": metrics["best_epoch"],
            },
        }
    if set(recipes) != {(m, p) for m in ("lstm", "cnn_lstm") for p in ("P0", "P1")}:
        raise ValueError("Freeze exactly four matched model/filter conditions")
    for model in ("lstm", "cnn_lstm"):
        a, b = [recipes[(model, p)] for p in ("P0", "P1")]
        for key in (
            "model",
            "features",
            "feature_statistics",
            "training",
            "budget",
            "cohort",
            "source",
            "versions",
            "scaler_sha256",
        ):
            if a["scientific"][key] != b["scientific"][key]:
                raise ValueError(f"Unmatched P0/P1 control: {key}")
        choices = [read_json(outputs(cfg, r["name"]) / "val.json") for r in (a, b)]
        if choices[0]["target_sha256"] != choices[1]["target_sha256"]:
            raise ValueError("The P0/P1 validation targets differ")
        highlights[model] = min(choices, key=ranking_key)["name"]
    lock = {
        "schema": SCHEMA,
        "cohort": manifest["cohort"],
        "manifest_sha256": file_hash(Path(cache) / "manifest.json"),
        "protocol": protocol(cfg),
        "scaler_sha256": file_hash(Path(cache) / "scaler.json"),
        "filter": cfg["preprocessing"],
        "runs": items,
        "highlighted_by_validation": highlights,
        "baselines": ["persistence", "repeat_window"],
        "synthetic": manifest["synthetic"],
        "used_test_for_selection": False,
    }
    if path.exists():
        existing = read_json(path)
        if {k: v for k, v in existing.items() if k != "frozen_at"} != lock:
            raise ValueError("Selection is already frozen; use a new study ID")
        return existing
    lock["frozen_at"] = datetime.now(timezone.utc).isoformat()
    write_json(path, lock)
    return lock


def final_evaluation(cfg, cache):
    lock = verify_lock(cfg, cache)
    cache, _ = prepare(cfg, include_test=True, lock=lock)
    path = study_dir(cfg) / "full" / "test_ledger.json"
    ledger = read_json(path) if path.exists() else {"lock_hash": digest(lock), "conditions": {}}
    if ledger["lock_hash"] != digest(lock):
        raise ValueError("Final-test ledger belongs to another selection")
    for name in list(lock["runs"]) + lock["baselines"]:
        receipt = outputs(cfg, name) / "test.json"
        previous = ledger["conditions"].get(name, {})
        if previous.get("state") == "completed":
            if previous["metrics_sha256"] != file_hash(receipt):
                raise ValueError("Completed test result changed")
            verify_evaluation(receipt.parent, "test")
            continue
        if previous.get("state") == "running" and (receipt.parent / "test_receipt.json").exists():
            verify_evaluation(receipt.parent, "test")
            saved = read_json(receipt)
            if (
                saved["cohort"] != lock["cohort"]
                or saved.get("checkpoint_sha256") != previous["checkpoint_sha256"]
            ):
                raise ValueError("Interrupted test result does not match the selection")
            ledger["conditions"][name].update(state="completed", metrics_sha256=file_hash(receipt))
            write_json(path, ledger)
            continue
        ledger["conditions"][name] = {
            "state": "running",
            "checkpoint_sha256": lock["runs"].get(name, {}).get("checkpoint_sha256"),
        }
        write_json(path, ledger)
        if name in lock["baselines"]:
            baseline(cfg, cache, name, "test")
        else:
            evaluate_run(cfg, cache, study_dir(cfg) / "full" / "checkpoints" / name, "test")
        ledger["conditions"][name].update(state="completed", metrics_sha256=file_hash(receipt))
        write_json(path, ledger)
    return ledger

remember_source("evaluation.txt")

### Training mRMR and validation permutation importance

In [ ]:
"""An isolated expanded-feature extension: train-only mRMR and validation permutation."""

from pathlib import Path

import numpy as np
import pandas as pd
import torch



def discretize(values, bins=8):
    unique = np.unique(values)
    if len(unique) <= bins:
        return np.searchsorted(unique, values)
    boundaries = np.unique(np.quantile(values, np.linspace(0, 1, bins + 1)[1:-1]))
    return np.searchsorted(boundaries, values, side="right")


def mutual_information(first, second):
    _, a = np.unique(first, return_inverse=True)
    _, b = np.unique(second, return_inverse=True)
    joint = np.zeros((a.max() + 1, b.max() + 1))
    np.add.at(joint, (a, b), 1)
    joint /= joint.sum()
    independent = joint.sum(0)[None, :] * joint.sum(1)[:, None]
    use = joint > 0
    return float(np.sum(joint[use] * np.log(joint[use] / independent[use])))


def mrmr_rank(features, target, names, groups):
    x = np.column_stack([discretize(features[:, i]) for i in range(features.shape[1])])
    y = np.column_stack([discretize(target[:, i]) for i in range(target.shape[1])])
    relevance = {
        g: float(
            np.mean(
                [
                    mutual_information(x[:, names.index(c)], y[:, j])
                    for c in columns
                    for j in range(y.shape[1])
                ]
            )
        )
        for g, columns in groups.items()
    }
    selected, rows = [], []
    while len(selected) < len(groups):
        scores = []
        previous = [c for g in selected for c in groups[g]]
        for group, columns in groups.items():
            if group in selected:
                continue
            redundancy = (
                float(
                    np.mean(
                        [
                            mutual_information(x[:, names.index(a)], x[:, names.index(b)])
                            for a in columns
                            for b in previous
                        ]
                    )
                )
                if previous
                else 0.0
            )
            scores.append((relevance[group] - redundancy, group, redundancy))
        score, group, redundancy = max(scores, key=lambda r: (r[0], -list(groups).index(r[1])))
        selected.append(group)
        rows.append(
            {
                "group": group,
                "rank": len(selected),
                "relevance_MI": relevance[group],
                "redundancy_MI": redundancy,
                "score": score,
            }
        )
    return pd.DataFrame(rows)


def save_selection(cfg, cache, method, ranking, names, groups, selected, details):
    path = (
        study_dir(cfg)
        / cfg["run_mode"]
        / "feature_selection"
        / details.get("architecture", "shared")
    )
    path.mkdir(parents=True, exist_ok=True)
    ranking.to_csv(path / f"{method}_ranking.csv", index=False)
    chosen = {c for group in selected for c in groups[group]}
    artifact = {
        "method": method,
        "cohort": read_json(Path(cache) / "manifest.json")["cohort"],
        "selected_groups": selected,
        "selected_features": [n for n in names if n in chosen],
        "groups": groups,
        "used_test": False,
        **details,
    }
    artifact["signature"] = digest(artifact)
    write_json(path / f"{method}.json", artifact)
    return artifact


def mrmr(cfg, cache):
    # No validation/test arrays or outcomes are opened for this ranking.
    arrays, names, info = feature_arrays(cfg, cache, "expanded", splits=("train",))
    waves = np.load(Path(cache) / "waves_train.npy", mmap_mode="r")
    h = cfg["data"]["history"]
    summaries = []
    for start in range(0, len(waves), 128):
        target = np.asarray(waves[start : start + 128, h:], dtype=np.float64)
        summaries.append(
            np.column_stack(
                [np.sqrt((target**2).mean(1)), np.sqrt((np.diff(target, axis=1) ** 2).mean(1))]
            )
        )
    ranking = mrmr_rank(arrays["train"], np.concatenate(summaries), names, info["groups"])
    selected = ranking.group.iloc[: cfg["features"]["selected_groups"]].tolist()
    return save_selection(
        cfg,
        cache,
        "mrmr",
        ranking,
        names,
        info["groups"],
        selected,
        {
            "fitted_split": "train",
            "target": "24 future RMS/difference-RMS summaries, training only",
            "estimator": "8-bin training quantile grouped MID",
            "used_validation_outcomes": False,
        },
    )


def permutation(cfg, cache, run):
    run = Path(run)
    record = read_json(run / "run.json")
    if record["feature_set"] != "expanded":
        raise ValueError("Permutation ranking needs a trained expanded-feature reference")
    arrays, names, info = feature_arrays(cfg, cache, "expanded")
    if names != record["selected_features"]:
        raise ValueError("Expanded feature order changed")
    dataset = ECGDataset(cache, cfg, "val", record["preprocessing"], arrays["val"])
    batches, device = loader(dataset, cfg), device_for(cfg)
    model = build_model(
        record["architecture"], len(names), cfg["data"]["horizon"], **record["config"]["model"]
    ).to(device)
    state = torch.load(run / "best.pt", map_location=device, weights_only=True)
    model.load_state_dict(state["model"])
    scaler = read_json(Path(cache) / "scaler.json")
    prediction, target = predict(model, batches, device)
    target = inverse_scale(target, scaler)
    patients = dataset.metadata.patient_id.to_numpy()
    baseline = metric_summary(inverse_scale(prediction, scaler), target, patients)[0][
        "patient_macro_mae_mV"
    ]
    rng, original = np.random.default_rng(cfg["training"]["seed"]), dataset.static.copy()
    unique = np.unique(patients)
    first = {p: np.flatnonzero(patients == p)[0] for p in unique}
    rows = []
    try:
        for group, columns in info["groups"].items():
            indices, changes = [names.index(c) for c in columns], []
            for _ in range(cfg["features"]["permutation_repeats"]):
                donors = dict(zip(unique, rng.permutation(unique), strict=True))
                source = [first[donors[p]] for p in patients]
                shuffled = original.copy()
                shuffled[:, indices] = original[source][:, indices]
                dataset.static = shuffled
                prediction, _ = predict(model, batches, device)
                score = metric_summary(inverse_scale(prediction, scaler), target, patients)[0][
                    "patient_macro_mae_mV"
                ]
                changes.append(score - baseline)
            rows.append(
                {
                    "group": group,
                    "importance_mV": float(np.mean(changes)),
                    "repeat_std_mV": float(np.std(changes)),
                }
            )
    finally:
        dataset.static = original
    ranking = pd.DataFrame(rows).sort_values(["importance_mV", "group"], ascending=[False, True])
    selected = (
        ranking.loc[ranking.importance_mV > 0, "group"]
        .iloc[: cfg["features"]["selected_groups"]]
        .tolist()
    )
    return save_selection(
        cfg,
        cache,
        "permutation",
        ranking,
        names,
        info["groups"],
        selected,
        {
            "architecture": record["architecture"],
            "fitted_split": "val",
            "source_checkpoint_sha256": file_hash(run / "best.pt"),
            "permutation_unit": "patient donors; first donor record; variables and flags moved together",
            "limitation": "first-record donor approximates within-patient variation; correlated variables can substitute",
            "interpretation": "predictive association, not causality",
        },
    )

remember_source("features.txt")

### Patient and temporal leakage checks

In [ ]:
"""Executable patient, temporal-boundary, and raw-training-scaler checks."""

from pathlib import Path

import numpy as np
import pandas as pd



def audit(cfg, cache):
    manifest = validate_cache(cache, cfg)
    frame = metadata(cfg["data"]["root"])
    patients = [set(frame.loc[frame.split.eq(s), "patient_id"]) for s in ("train", "val", "test")]
    assert all(not patients[i] & patients[j] for i in range(3) for j in range(i + 1, 3))
    checks = ["official patient groups are disjoint"]
    for split in ("train", "val"):
        rows = pd.read_csv(Path(cache) / f"metadata_{split}.csv")
        assert rows.ecg_id.is_unique
        assert manifest["splits"][split]["records"] == manifest["splits"][split]["windows"]
    checks.append("exactly one 1,000-sample record window in training and validation")
    raw = np.load(Path(cache) / "waves_train.npy", mmap_mode="r")
    scaler = read_json(Path(cache) / "scaler.json")
    fitted = waveform_scaler(raw, cfg["data"]["history"])
    np.testing.assert_array_equal(scaler["mean"], fitted["mean"])
    np.testing.assert_array_equal(scaler["std"], fitted["std"])
    checks.append("scaler refits exactly from raw training history only")
    wave = np.load(Path(cache) / "waves_val.npy", mmap_mode="r")[0].copy()
    changed = wave.copy()
    h = cfg["data"]["history"]
    changed[h:] += 10000
    np.testing.assert_array_equal(wave[:h], changed[:h])
    np.testing.assert_array_equal(filter_history(wave[:h], cfg), filter_history(changed[:h], cfg))
    np.testing.assert_allclose(observed_rr(wave[:h]), observed_rr(changed[:h]), equal_nan=True)
    checks.append("future mutation cannot alter raw/filtered histories or observed RR features")
    prefix = h // 2
    modified_history = wave[:h].copy()
    modified_history[prefix:] += 1000
    np.testing.assert_array_equal(
        filter_history(wave[:h], cfg)[:prefix], filter_history(modified_history, cfg)[:prefix]
    )
    checks.append("causal filter prefix is independent of later history samples")
    result = {
        "passed": True,
        "cohort": manifest["cohort"],
        "checks": checks,
        "synthetic": manifest["synthetic"],
        "test_signal_read": False,
    }
    write_json(study_dir(cfg) / cfg["run_mode"] / "leakage_audit.json", result)
    return result

remember_source("audit.txt")

### Diagnostic and forecast figures

In [ ]:
"""Figures with physical axes, reproducible examples, and explicit captions."""

from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import signal



def save(fig, path, shows, conclusion):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.tight_layout()
    fig.savefig(path, dpi=140)
    plt.close(fig)
    path.with_suffix(".md").write_text(f"{shows}\n\n{conclusion}\n", encoding="utf-8")


def diagnostic_figures(cfg, cache):
    out = study_dir(cfg) / cfg["run_mode"] / "figures" / "data"
    frame = metadata(cfg["data"]["root"])
    counts = frame.groupby("strat_fold").agg(
        records=("ecg_id", "size"), patients=("patient_id", "nunique")
    )
    fig, ax = plt.subplots(figsize=(10, 4))
    counts.plot.bar(ax=ax)
    ax.set(xlabel="Official fold", ylabel="Count", title="Official record and patient assignments")
    save(
        fig,
        out / "fold_counts.png",
        "The chart shows metadata counts in the official folds.",
        "No patient occurs in more than one fold; fold 10 is reserved and no test scores are used here.",
    )
    raw = np.load(Path(cache) / "waves_val.npy", mmap_mode="r")
    rows = pd.read_csv(Path(cache) / "metadata_val.csv")
    index = int(np.argmin(rows.ecg_id.to_numpy()))
    wave, h = raw[index], cfg["data"]["history"]
    observed, filtered = wave[:h], filter_history(wave[:h], cfg)
    for kind in ("original", "filtered_history", "morphology_zoom", "psd"):
        fig, axes = plt.subplots(4, 3, figsize=(14, 11))
        for lead, ax in enumerate(axes.flat):
            if kind == "original":
                ax.plot(np.arange(1000) / 100, wave[:, lead], lw=0.7)
                ax.axvline(h / 100, color="black", ls="--")
                ax.set(xlabel="Time (s)", ylabel="mV")
            elif kind == "psd":
                for values, label in (
                    (observed, "Raw history"),
                    (filtered, "Causal filtered history"),
                ):
                    frequency, density = signal.welch(values[:, lead], fs=100, nperseg=min(256, h))
                    ax.semilogy(frequency, np.maximum(density, 1e-15), label=label)
                ax.set(xlabel="Hz", ylabel="mV²/Hz")
            else:
                time = np.arange(h) / 100
                ax.plot(time, observed[:, lead], label="Raw history", lw=0.8)
                ax.plot(time, filtered[:, lead], label="Causal filtered history", lw=0.8)
                ax.set(xlabel="Observed time (s)", ylabel="mV")
                if kind == "morphology_zoom":
                    center = np.argmax(np.abs(observed[:, 1] - np.median(observed[:, 1]))) / 100
                    ax.set_xlim(max(0, center - 0.25), min(h / 100, center + 0.25))
            ax.set_title(LEADS[lead])
        if kind != "original":
            axes.flat[0].legend(fontsize=7)
        fig.suptitle(f"ECG {int(rows.iloc[index].ecg_id)}: {kind.replace('_', ' ')}", y=1.01)
        save(
            fig,
            out / f"{kind}.png",
            f"This figure shows all twelve leads for the lowest-ID validation recording ({kind}).",
            "Filtering changes observed voltages and phase; these plots alone do not establish better forecasts or clinical morphology preservation.",
        )
    write_json(
        out / "filter_distortion.json",
        {
            "ecg_id": int(rows.iloc[index].ecg_id),
            "history_only_rmse_mV_by_lead": np.sqrt(
                ((filtered - observed) ** 2).mean(axis=0)
            ).tolist(),
            "future_used": False,
            "filter_prototype_order": cfg["preprocessing"]["order"],
            "bandpass_total_order": 2 * cfg["preprocessing"]["order"],
        },
    )


def run_figures(cfg, cache, run, split="val"):
    run = Path(run)
    metrics_dir = study_dir(cfg) / cfg["run_mode"] / "metrics" / run.name
    out = study_dir(cfg) / cfg["run_mode"] / "figures" / run.name
    if (run / "history.csv").exists():
        history = pd.read_csv(run / "history.csv")
        selected = read_json(metrics_dir / f"{split}.json")["best_epoch"]
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        for prefix, label in (("train", "Training"), ("val", "Validation")):
            axes[0].plot(history.epoch, history[prefix + "_mse_normalized"], label=label)
            axes[1].plot(history.epoch, history[prefix + "_patient_macro_mae_mV"], label=label)
        for ax in axes:
            ax.axvline(selected, color="black", ls="--", label=f"Selected epoch {selected}")
            ax.set_xlabel("Actual completed epoch")
            ax.legend(fontsize=8)
        axes[0].set_ylabel("Normalized MSE")
        axes[1].set_ylabel("Patient-macro MAE (mV)")
        save(
            fig,
            out / "learning_curves.png",
            "The curves show completed training epochs and the validation-selected checkpoint.",
            "A lower training loss alone does not demonstrate better held-out forecasts; selection uses validation patient-macro MAE.",
        )
    result = read_json(metrics_dir / f"{split}.json")
    with np.load(metrics_dir / f"{split}_predictions.npz") as archive:
        ids, target, prediction = (archive[key] for key in ("ecg_id", "target", "prediction"))
    wave = np.load(Path(cache) / f"waves_{split}.npy", mmap_mode="r")
    h = cfg["data"]["history"]
    for kind, ecg_id in (
        ("pre_specified", result["pre_specified_ecg_id"]),
        ("median_error", result["median_error_ecg_id"]),
    ):
        index = int(np.flatnonzero(ids == ecg_id)[0])
        fig, axes = plt.subplots(4, 3, figsize=(14, 11))
        for lead, ax in enumerate(axes.flat):
            ax.plot(np.arange(h) / 100, wave[index, :h, lead], label="Observed", lw=0.65)
            future = np.arange(h, 1000) / 100
            ax.plot(future, target[index, :, lead], label="Actual future", lw=0.8)
            ax.plot(future, prediction[index, :, lead], label="Predicted future", lw=0.8)
            ax.axvline(h / 100, color="black", ls="--", lw=0.7)
            ax.set(title=LEADS[lead], xlabel="Time (s)", ylabel="mV")
        axes.flat[0].legend(fontsize=7)
        fig.suptitle(f"{split}: ECG {ecg_id}; {kind.replace('_', ' ')} example", y=1.01)
        save(
            fig,
            out / f"{split}_{kind}_12_leads.png",
            f"Observed history, raw future truth, and forecast are shown separately for each lead; this is the {kind.replace('_', ' ')} case.",
            "This example illustrates waveform error, not clinical validity or performance on every patient.",
        )
    lead_metrics = pd.read_csv(metrics_dir / f"{split}_leads.csv")
    fig, ax = plt.subplots(figsize=(11, 4))
    lead_metrics.set_index("lead")[["mae_mV", "rmse_mV"]].plot.bar(ax=ax)
    ax.set(ylabel="Error (mV)", xlabel="ECG lead")
    save(
        fig,
        out / f"{split}_per_lead_errors.png",
        "MAE and RMSE are shown in mV for each of the twelve leads.",
        "Differences show absolute voltage errors; low error does not prove accurate peak timing.",
    )

remember_source("figures.txt")

### Evidence comparison, reporting and export

In [ ]:
"""Aggregate only traceable measurements and identify controlled comparisons."""

from itertools import combinations

import pandas as pd


RESULT_COLUMNS = [
    "name",
    "state",
    "split",
    "architecture",
    "preprocessing",
    "feature_set",
    "mae_mV",
    "rmse_mV",
    "patient_macro_mae_mV",
    "mse_mV2",
    "r2",
    "pearson_mean",
    "records",
    "patients",
    "seed",
    "lr",
    "hidden",
    "epochs_completed",
    "best_epoch",
    "runtime_seconds",
]


def collect(cfg):
    root = study_dir(cfg) / cfg["run_mode"]
    rows, lead_frames, records = [], [], {}
    for directory in sorted((root / "checkpoints").glob("*")):
        if not (directory / "run.json").exists():
            continue
        record = read_json(directory / "run.json")
        row = {
            "name": directory.name,
            "state": "training incomplete",
            "architecture": record["architecture"],
            "preprocessing": record["preprocessing"],
            "feature_set": record["feature_set"],
            "parameters": record["parameters"],
            "seed": record["config"]["training"]["seed"],
            "lr": record["config"]["training"]["lr"],
            "hidden": record["config"]["model"]["hidden"],
            "smoke": record["smoke"],
        }
        if (directory / "status.json").exists():
            state = read_json(directory / "status.json")
            row.update(
                state=state["state"],
                epochs_completed=state["epochs_completed"],
                runtime_seconds=state["runtime_seconds"],
            )
        for split in ("val", "test"):
            path = root / "metrics" / directory.name / f"{split}.json"
            if not path.exists():
                continue
            metrics = read_json(path)
            try:
                verify_evaluation(path.parent, split)
                evidence_valid = True
            except (ValueError, FileNotFoundError):
                evidence_valid = False
            valid = (
                record["signature"] == digest(record["scientific"])
                and evidence_valid
                and metrics["cohort"] == record["scientific"]["cohort"]
                and metrics["run_signature"] == record["signature"]
                and metrics["protocol"] == protocol(cfg)
                and (directory / "best.pt").exists()
                and metrics["checkpoint_sha256"] == file_hash(directory / "best.pt")
                and (directory / "cohort_manifest.json").exists()
                and (directory / "scaler.json").exists()
                and file_hash(directory / "scaler.json") == record["scientific"]["scaler_sha256"]
            )
            entry = {**row, "split": split}
            if not valid:
                entry["state"] = "incompatible or unavailable provenance"
            elif row["state"] != "completed":
                entry["state"] = "training incomplete"
            elif metrics["smoke"]:
                if not metrics["synthetic"]:
                    entry.update(metrics)
                entry["state"] = "software smoke check passed; excluded from research comparison"
            else:
                entry.update(metrics, state="measured")
                lead_frames.append(
                    pd.read_csv(path.with_name(f"{split}_leads.csv")).assign(
                        name=directory.name, split=split
                    )
                )
                records[(directory.name, split)] = record
            rows.append(entry)
        if not any(
            (root / "metrics" / directory.name / f"{s}.json").exists() for s in ("val", "test")
        ):
            rows.append(row)
    for name in ("persistence", "repeat_window"):
        for split in ("val", "test"):
            path = root / "metrics" / name / f"{split}.json"
            if path.exists():
                result = read_json(path)
                try:
                    verify_evaluation(path.parent, split)
                except (ValueError, FileNotFoundError):
                    rows.append(
                        {
                            "name": name,
                            "split": split,
                            "state": "incompatible or unavailable provenance",
                        }
                    )
                    continue
                if result["smoke"] and not result["synthetic"]:
                    rows.append(
                        {
                            **result,
                            "state": "software smoke check passed; excluded from research comparison",
                        }
                    )
                elif not result["smoke"] and result["protocol"] == protocol(cfg):
                    rows.append(
                        {**result, "state": "measured", "epochs_completed": 0, "best_epoch": None}
                    )
                    lead_frames.append(
                        pd.read_csv(path.with_name(f"{split}_leads.csv")).assign(
                            name=name, split=split
                        )
                    )
    table = pd.DataFrame(rows).reindex(
        columns=list(dict.fromkeys(RESULT_COLUMNS + [k for r in rows for k in r]))
    )
    leads = (
        pd.concat(lead_frames, ignore_index=True)
        if lead_frames
        else pd.DataFrame(columns=["name", "split", "lead", "mae_mV", "rmse_mV", "pearson"])
    )
    return table, leads, records


def paired_comparisons(cfg, table, records):
    output = []
    candidates = table.loc[table.state.eq("measured")].to_dict("records")
    for a, b in combinations(candidates, 2):
        if (
            a["split"] != b["split"]
            or a["cohort"] != b["cohort"]
            or a["target_sha256"] != b["target_sha256"]
        ):
            continue
        ra, rb = records.get((a["name"], a["split"])), records.get((b["name"], b["split"]))
        contrast = None
        if ra is None or rb is None:
            if (ra is None) != (rb is None):
                contrast = "neural pipeline versus raw-history baseline"
                if rb is None:
                    a, b = b, a
        else:
            sa, sb = ra["scientific"], rb["scientific"]
            controls = (
                "cohort",
                "model",
                "training",
                "budget",
                "source",
                "versions",
                "scaler_sha256",
            )
            same_model = a["architecture"] == b["architecture"]
            same_features = sa["features"] == sb["features"]
            same_filter = a["preprocessing"] == b["preprocessing"]
            matched = all(sa[k] == sb[k] for k in controls)
            if not matched:
                if cfg["run_mode"] == "full" and not same_model and same_filter:
                    contrast = "selected pipelines (architecture and recipe may differ)"
                elif (
                    same_model
                    and same_filter
                    and same_features
                    and sa["model"] == sb["model"]
                    and (
                        {k: v for k, v in sa["training"].items() if k != "lr"}
                        == {k: v for k, v in sb["training"].items() if k != "lr"}
                    )
                    and all(sa[k] == sb[k] for k in controls if k != "training")
                ):
                    contrast = "learning-rate comparison"
            elif same_model and same_features and not same_filter:
                contrast = "P0 versus P1"
                if a["preprocessing"] == "P1":
                    a, b = b, a
            elif same_model and same_filter and not same_features:
                contrast = "static-feature ablation"
                priority = {"F0": 0, "F1": 1, "expanded": 2, "mrmr": 3, "permutation": 3}
                if priority.get(a["feature_set"], 2) > priority.get(b["feature_set"], 2):
                    a, b = b, a
            elif not same_model and same_features and same_filter:
                contrast = "matched architecture comparison"
        if contrast is None:
            continue
        root = study_dir(cfg) / cfg["run_mode"] / "metrics"
        first = pd.read_csv(root / a["name"] / f"{a['split']}_records.csv")
        second = pd.read_csv(root / b["name"] / f"{b['split']}_records.csv")
        comparison = bootstrap_pair(
            first, second, cfg["evaluation"]["bootstrap_samples"], cfg["training"]["seed"]
        )
        denominator = a["patient_macro_mae_mV"]
        output.append(
            {
                "A": a["name"],
                "B": b["name"],
                "split": a["split"],
                "contrast": contrast,
                "percentage_B_minus_A": 100
                * (b["patient_macro_mae_mV"] - denominator)
                / denominator
                if denominator > 1e-12
                else None,
                **comparison,
            }
        )
    return pd.DataFrame(
        output,
        columns=[
            "A",
            "B",
            "split",
            "contrast",
            "percentage_B_minus_A",
            "difference_B_minus_A_mV",
            "ci95_low",
            "ci95_high",
            "patients",
            "bootstrap_samples",
            "conclusion",
        ],
    )

"""Build a report from saved evidence; never train or invent missing results."""

from pathlib import Path
from zipfile import ZIP_DEFLATED, ZipFile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


SUMMARY_ORDER = [
    "Dataset",
    "Patient-safe folds",
    "Preprocessing",
    "LSTM",
    "CNN-LSTM",
    "Training",
    "RMSE/MAE",
    "Twelve-lead prediction",
    "Evidence-based comparison",
]


def markdown_table(frame):
    if frame.empty:
        return "Pending: no completed comparable research measurements."

    def cell(value):
        if pd.isna(value):
            return "pending"
        return f"{value:.6g}" if isinstance(value, float) else str(value).replace("|", "/")

    lines = [
        "| " + " | ".join(frame.columns) + " |",
        "| " + " | ".join(["---"] * len(frame.columns)) + " |",
    ]
    lines += [
        "| " + " | ".join(cell(v) for v in row) + " |"
        for row in frame.itertuples(index=False, name=None)
    ]
    return "\n".join(lines)


def comparison_figures(cfg, table, paired):
    measured = table.loc[table.state.eq("measured")].copy()
    if measured.empty:
        return
    out = study_dir(cfg) / cfg["run_mode"] / "figures" / "comparisons"
    for split in [s for s in sorted(measured.split.unique()) if s == "test"]:
        selected = measured.loc[measured.split.eq(split)]
        fig, ax = plt.subplots(figsize=(max(9, len(selected) * 0.7), 5))
        selected = selected.assign(
            label=selected.architecture + "/" + selected.preprocessing + "/" + selected.feature_set
        )
        selected.set_index("label")[["mae_mV", "rmse_mV"]].plot.bar(ax=ax)
        ax.set(ylabel="Error (mV)", xlabel="Recorded experiment")
        save(
            fig,
            out / f"{split}_errors.png",
            f"Measured {split} MAE/RMSE are shown for the same waveform task.",
            "Rows with different feature or optimizer settings compare complete recipes; only controlled pairs isolate one factor.",
        )
    for split in ("val", "test"):
        for kind, contrast in (
            ("preprocessing", "P0 versus P1"),
            ("features", "static-feature ablation"),
        ):
            pairs = paired.loc[paired.split.eq(split) & paired.contrast.eq(contrast)]
            if pairs.empty:
                continue
            fig, ax = plt.subplots(figsize=(10, max(4, len(pairs) * 0.5)))
            values = pairs.difference_B_minus_A_mV.to_numpy()
            errors = [
                np.maximum(0, values - pairs.ci95_low.to_numpy()),
                np.maximum(0, pairs.ci95_high.to_numpy() - values),
            ]
            labels = [f"{a} → {b}" for a, b in zip(pairs.A, pairs.B, strict=True)]
            ax.barh(np.arange(len(pairs)), values, xerr=errors)
            ax.set(
                yticks=np.arange(len(pairs)),
                yticklabels=labels,
                xlabel="B − A patient-macro MAE (mV)",
            )
            ax.axvline(0, color="black", lw=0.8)
            save(
                fig,
                out / f"{kind}_{split}.png",
                f"Matched {contrast} differences and paired patient-bootstrap intervals are shown.",
                "Negative differences favor B; intervals crossing zero are inconclusive, and single-seed intervals do not capture training variability.",
            )
    lr = measured.loc[measured.split.eq("val") & measured.lr.notna()]
    if len(lr):
        fig, ax = plt.subplots(figsize=(10, 5))
        for (model, filtering, features, hidden), group in lr.groupby(
            ["architecture", "preprocessing", "feature_set", "hidden"]
        ):
            if group.lr.nunique() > 1:
                group = group.sort_values("lr")
                ax.plot(
                    group.lr,
                    group.patient_macro_mae_mV,
                    marker="o",
                    label=f"{model}/{filtering}/{features}/h{hidden:g}",
                )
        if ax.lines:
            ax.set(xscale="log", xlabel="Learning rate", ylabel="Validation patient-macro MAE (mV)")
            ax.legend(fontsize=7)
            save(
                fig,
                out / "learning_rates.png",
                "Validation results compare the tested learning rates within fixed recipes.",
                "The selected rate is best among these measured candidates, not a proven global optimum.",
            )
        else:
            plt.close(fig)


def generate(cfg):
    root = study_dir(cfg) / cfg["run_mode"]
    root.mkdir(parents=True, exist_ok=True)
    table, leads, records = collect(cfg)
    paired = paired_comparisons(cfg, table, records)
    table.to_csv(root / "experiment_results.csv", index=False)
    leads.to_csv(root / "per_lead_metrics.csv", index=False)
    paired.to_csv(root / "paired_comparisons.csv", index=False)
    comparison_figures(cfg, table, paired)
    manifest_path = root / "cohort.json"
    manifest = read_json(manifest_path) if manifest_path.exists() else None
    counts = pd.DataFrame(manifest["splits"]).T if manifest else pd.DataFrame()
    measured = table.loc[table.state.eq("measured")]
    smoke = (
        table.loc[
            table.state.eq("software smoke check passed; excluded from research comparison")
            & table["synthetic"].eq(False)
        ]
        if "synthetic" in table
        else table.iloc[:0]
    )
    metrics_columns = [
        "name",
        "split",
        "mae_mV",
        "rmse_mV",
        "patient_macro_mae_mV",
        "mse_mV2",
        "r2",
        "pearson_mean",
    ]
    hardware = [read_json(p)["environment"] for p in (root / "checkpoints").glob("*/run.json")]
    model_rows = []
    for path in (root / "checkpoints").glob("*/run.json"):
        run = read_json(path)
        model_rows.append(
            {
                "model": run["architecture"],
                "filter": run["preprocessing"],
                "features": run["feature_set"],
                "static_values": len(run["selected_features"]),
                "hidden": run["config"]["model"]["hidden"],
                "layers": run["config"]["model"]["depth"],
                "active_parameters": run["parameters"],
                "output": f"[B,{cfg['data']['horizon']},12]",
            }
        )
    model_table = pd.DataFrame(
        model_rows,
        columns=[
            "model",
            "filter",
            "features",
            "static_values",
            "hidden",
            "layers",
            "active_parameters",
            "output",
        ],
    )
    status_columns = ["name", "state", "epochs_completed", "best_epoch", "runtime_seconds"]
    sections = {
        "Dataset": f"PTB-XL v1.0.3; source: https://physionet.org/content/ptb-xl/1.0.3/. Data root: `{cfg['data']['root']}/records100`. WFDB reads physical mV, 100 Hz, twelve leads, 1,000 samples per record. History/target: `{cfg['data']['history']}/{cfg['data']['horizon']}`; batched input `[B,{cfg['data']['history']},12]`, output `[B,{cfg['data']['horizon']},12]`. This is within-record waveform forecasting, not clinical longitudinal prediction.",
        "Patient-safe folds": "Official folds 1–8 train, 9 validate, 10 test; patients remain disjoint. Selected whole patients retain all eligible ECGs.\n\n"
        + markdown_table(counts.reset_index(names="split"))
        + "\n\nReserved fold 10 metadata counts: "
        + (
            str({k: v for k, v in manifest["reserved_test"].items() if k != "prepared"})
            if manifest
            else "preparation pending"
        )
        + "\n\nFinal-test ledger: "
        + (
            str(read_json(root / "test_ledger.json")["conditions"])
            if (root / "test_ledger.json").exists()
            else "pending / not enabled"
        ),
        "Preprocessing": "P0 uses raw history; P1 uses causal 0.5–40 Hz SOS Butterworth filtering on observed history only (prototype N=4; bandpass order 8). Both use identical raw-training-history z-scores and predict the same original-mV future. Filtering may distort ST segments, QRS shape, phase, and initial samples; it is a hypothesis. History-only distortion/PSD figures do not establish physiological accuracy.",
        "LSTM": "Independent `LSTMForecaster`: optional early static embedding → stacked LSTM → final hidden state → MLP → all future samples. Exact active parameter counts and model settings are in each run.json; see the model and metrics code cells.",
        "CNN-LSTM": "Independent `CNNLSTMForecaster`: optional static channels → Conv1d(7,32) → ReLU → Conv1d(5,64) → ReLU → AvgPool(2) → LSTM → MLP. The primary 700-step history becomes 350 recurrent steps. No attention or contrastive projector is active.",
        "Training": "AdamW, normalized MSE, fixed learning rates, gradient clipping, seed 42; validation patient-macro MAE selects checkpoints. No early stopping truncates the budget.\n\n"
        + markdown_table(table.reindex(columns=status_columns))
        + "\n\nRecorded environments: `"
        + str(hardware)
        + "`. Full 120-epoch Colab runs remain pending unless their completed artifacts are present.",
        "RMSE/MAE": "MAE = mean(|prediction − truth|), in mV; RMSE = sqrt(mean((prediction − truth)²)), in mV; MSE is mV². RMSE applies to both regression architectures and is not an LSTM gate. Patient-macro MAE averages records within each patient, then weights patients equally.\n\n"
        + markdown_table(measured.reindex(columns=metrics_columns)),
        "Twelve-lead prediction": "Saved figures show each lead separately with observed history, original future truth, forecast, physical axes, and the forecast boundary. The pre-specified example is selected by metadata ID before errors; the median-error example is labelled. No fabricated peak markers or unvalidated P/Q/R/S/T delineation are used.\n\n"
        + markdown_table(leads.head(24)),
        "Evidence-based comparison": markdown_table(paired)
        + "\n\nNegative B−A favors B. Intervals resample patients and are descriptive after validation selection, using one seed without multiplicity correction. Medium/full cohorts and different horizons are not pooled. Low voltage error does not establish clinical validity; 100-Hz resolution and filter/morphology limitations remain. ARIMA is deferred. Feature-selection execution is pending unless feature_extension.json exists. Synthetic and pilot runs are software checks, not research performance evidence.",
    }
    for model, heading in (("lstm", "LSTM"), ("cnn_lstm", "CNN-LSTM")):
        sections[heading] += "\n\n" + markdown_table(
            model_table.loc[model_table.model.eq(model)].drop_duplicates()
        )
    if not smoke.empty:
        sections["RMSE/MAE"] += (
            "\n\nSoftware pilot diagnostics (excluded from research rankings):\n\n"
            + markdown_table(smoke.reindex(columns=metrics_columns))
        )
    filter_comparisons = paired.loc[paired.split.eq("val") & paired.contrast.eq("P0 versus P1")]
    sections["Preprocessing"] += "\n\nMeasured validation filter effects:\n\n" + markdown_table(
        filter_comparisons
    )
    milestone_rows = []
    for path in (root / "metrics").glob("*_through_*/val.json"):
        item = read_json(path)
        if not item["smoke"]:
            milestone_rows.append(
                {k: item[k] for k in ("name", "best_epoch", "patient_macro_mae_mV", "rmse_mV")}
            )
    sections["Training"] += "\n\n60/120 checkpoint evidence:\n\n" + markdown_table(
        pd.DataFrame(milestone_rows)
    )
    if manifest and manifest["synthetic"]:
        warning = "**SYNTHETIC SOFTWARE FIXTURE — no PTB-XL performance claims.**\n\n"
    elif cfg["run_mode"] == "pilot":
        warning = "**REAL-DATA SOFTWARE PILOT — numerical diagnostics are excluded from research rankings.**\n\n"
    else:
        warning = ""
    text = (
        "# PTB-XL research results\n\n"
        + warning
        + "\n\n".join(f"## {name}\n\n{sections[name]}" for name in SUMMARY_ORDER)
    )
    (root / "RESULTS.md").write_text(text + "\n", encoding="utf-8")
    write_json(
        root / "summary.json", {"order": SUMMARY_ORDER, "sections": sections, "warning": warning}
    )
    return root / "RESULTS.md"


def bundle(cfg):
    root = study_dir(cfg)
    destination = root / "PTBXL_Research_Results.zip"
    temporary = destination.with_suffix(".tmp")
    with ZipFile(temporary, "w", ZIP_DEFLATED) as archive:
        for path in sorted(root.rglob("*")):
            if path.is_file() and path.suffix not in (".zip", ".tmp", ".lock"):
                archive.write(path, "results/" + path.relative_to(root).as_posix())
        archive.writestr("configuration.yaml", yaml.safe_dump(cfg, sort_keys=False))
        archive.writestr("default_configuration.yaml", DEFAULT_CONFIG_YAML.strip() + "\n")
    temporary.replace(destination)
    return destination

remember_source("reporting.txt")

### Pilot, screening, tuning and full training

In [ ]:
"""Explicit finite experiment stages, reusing the same training/evaluation functions."""

import copy



def condition(
    cfg, cache, model, filtering, features, *, resume=False, lr=None, hidden=None, selection=None
):
    budget = (
        cfg["training"]["pilot_epochs"]
        if cfg["run_mode"] == "pilot"
        else (
            cfg["training"]["full_epochs"]
            if cfg["run_mode"] == "full"
            else cfg["training"]["epochs"]
        )
    )
    rate, size = lr or cfg["training"]["lr"], hidden or cfg["model"]["hidden"]
    name = f"{model}_{filtering}_{features}_lr{rate:g}_h{size}_e{budget}"
    run = study_dir(cfg) / cfg["run_mode"] / "checkpoints" / name
    complete = (run / "status.json").exists() and read_json(run / "status.json")[
        "state"
    ] == "completed"
    # Completed runs may be reused, but train_run first checks the full scientific signature.
    run = train_run(
        cfg,
        cache,
        model,
        filtering,
        features,
        name=name,
        lr=rate,
        hidden=size,
        resume=complete or (resume and (run / "last.pt").exists()),
        selection=selection,
    )
    receipt = outputs(cfg, run.name) / "val.json"
    if not complete or not receipt.exists():
        evaluate_run(cfg, cache, run)
    else:
        verify_evaluation(receipt.parent, "val")
        result = read_json(receipt)
        if result["checkpoint_sha256"] != file_hash(run / "best.pt"):
            raise ValueError("Completed run's validation evidence does not match its checkpoint")
    return run, read_json(receipt)


def smoke(cfg, resume=False):
    cfg = copy.deepcopy(cfg)
    cfg["run_mode"] = "pilot"
    cache, manifest = prepare(cfg)
    audit(cfg, cache)
    runs = []
    for model in ("lstm", "cnn_lstm"):
        for filtering in ("P0", "P1"):
            run, _ = condition(cfg, cache, model, filtering, "F0", resume=resume)
            runs.append(run)
        run, _ = condition(cfg, cache, model, "P0", "F1", resume=resume)
        runs.append(run)
    for name in ("persistence", "repeat_window"):
        baseline(cfg, cache, name)
    diagnostic_figures(cfg, cache)
    for run in (runs[0], runs[3]):
        run_figures(cfg, cache, run)
    result = {
        "passed": True,
        "cohort": manifest["cohort"],
        "splits": manifest["splits"],
        "runs": [r.name for r in runs],
        "epochs_each": cfg["training"]["pilot_epochs"],
        "test_waveforms_opened": False,
        "research_performance_claim": False,
        "synthetic": manifest["synthetic"],
    }
    write_json(study_dir(cfg) / "pilot" / "smoke_checks.json", result)
    return result


def screen(cfg, resume=False):
    cfg = copy.deepcopy(cfg)
    cfg["run_mode"] = "medium"
    cache, manifest = prepare(cfg)
    audit(cfg, cache)
    selected = {}
    for model in ("lstm", "cnn_lstm"):
        candidates = []
        for filtering in ("P0", "P1"):
            _, result = condition(cfg, cache, model, filtering, "F0", resume=resume)
            candidates.append(result)
        best_filter = min(candidates, key=ranking_key)
        _, conditioned = condition(
            cfg, cache, model, best_filter["preprocessing"], "F1", resume=resume
        )
        best = min([best_filter, conditioned], key=ranking_key)
        selected[model] = {
            "preprocessing": best["preprocessing"],
            "feature_set": best["feature_set"],
            "run": best["name"],
            "lr": cfg["training"]["lr"],
            "hidden": cfg["model"]["hidden"],
        }
    result = {"cohort": manifest["cohort"], "selected": selected, "used_test": False}
    write_json(study_dir(cfg) / "medium" / "screening_selection.json", result)
    for name in ("persistence", "repeat_window"):
        baseline(cfg, cache, name)
    diagnostic_figures(cfg, cache)
    return result


def tune(cfg, resume=False):
    cfg = copy.deepcopy(cfg)
    cfg["run_mode"] = "medium"
    cache, manifest = prepare(cfg)
    previous = read_json(study_dir(cfg) / "medium" / "screening_selection.json")
    if previous["cohort"] != manifest["cohort"]:
        raise ValueError("Screening used a different cohort")
    selected = {}
    for model, recipe in previous["selected"].items():
        candidates = []
        for rate in cfg["training"]["learning_rates"]:
            run, result = condition(
                cfg,
                cache,
                model,
                recipe["preprocessing"],
                recipe["feature_set"],
                resume=resume,
                lr=rate,
            )
            candidates.append((run, result))
        if cfg["training"]["tune_hidden_128"]:
            best_run, _ = min(candidates, key=lambda x: ranking_key(x[1]))
            rate = read_json(best_run / "run.json")["config"]["training"]["lr"]
            candidates.append(
                condition(
                    cfg,
                    cache,
                    model,
                    recipe["preprocessing"],
                    recipe["feature_set"],
                    resume=resume,
                    lr=rate,
                    hidden=128,
                )
            )
        run, result = min(candidates, key=lambda x: ranking_key(x[1]))
        record = read_json(run / "run.json")
        selected[model] = {
            "preprocessing": recipe["preprocessing"],
            "feature_set": recipe["feature_set"],
            "lr": record["config"]["training"]["lr"],
            "hidden": record["config"]["model"]["hidden"],
            "validation_run": result["name"],
        }
    result = {"cohort": manifest["cohort"], "selected": selected, "used_test": False}
    write_json(study_dir(cfg) / "medium" / "tuning_selection.json", result)
    return result


def selected_recipes(cfg):
    path = study_dir(cfg) / "medium" / "tuning_selection.json"
    if not path.exists():
        raise ValueError("Complete medium screening and tuning before extensions/full training")
    result = read_json(path)
    if result.get("used_test") is not False:
        raise ValueError("Selection must use validation only")
    medium = copy.deepcopy(cfg)
    medium["run_mode"] = "medium"
    _, current = prepare(medium)
    if result["cohort"] != current["cohort"]:
        raise ValueError("Selected recipes used a different source/cohort/protocol")
    for recipe in result["selected"].values():
        run = study_dir(cfg) / "medium" / "checkpoints" / recipe["validation_run"]
        saved = read_json(run / "run.json")
        actual = (
            saved["preprocessing"],
            saved["feature_set"],
            saved["config"]["training"]["lr"],
            saved["config"]["model"]["hidden"],
        )
        if actual != (
            recipe["preprocessing"],
            recipe["feature_set"],
            recipe["lr"],
            recipe["hidden"],
        ):
            raise ValueError("Selected recipe no longer matches its validation run")
    return result["selected"]


def feature_extension(cfg, resume=False):
    cfg = copy.deepcopy(cfg)
    cfg["run_mode"] = "medium"
    cache, manifest = prepare(cfg)
    saved = read_json(study_dir(cfg) / "medium" / "tuning_selection.json")
    if saved["cohort"] != manifest["cohort"]:
        raise ValueError("Tuning used a different cohort")
    ranking = mrmr(cfg, cache)
    completed = []
    for model, recipe in saved["selected"].items():
        expanded, _ = condition(
            cfg,
            cache,
            model,
            recipe["preprocessing"],
            "expanded",
            resume=resume,
            lr=recipe["lr"],
            hidden=recipe["hidden"],
        )
        permuted = permutation(cfg, cache, expanded)
        for method, selection in (("mrmr", ranking), ("permutation", permuted)):
            run, _ = condition(
                cfg,
                cache,
                model,
                recipe["preprocessing"],
                method,
                resume=resume,
                lr=recipe["lr"],
                hidden=recipe["hidden"],
                selection=selection,
            )
            completed.append(run.name)
    result = {"completed": completed, "cohort": manifest["cohort"], "used_test": False}
    write_json(study_dir(cfg) / "medium" / "feature_extension.json", result)
    return result


def full(cfg, resume=False):
    cfg = copy.deepcopy(cfg)
    cfg["run_mode"] = "full"
    device_for(cfg)
    recipes = selected_recipes(cfg)
    cache, _ = prepare(cfg)
    audit(cfg, cache)
    for name in ("persistence", "repeat_window"):
        baseline(cfg, cache, name)
    diagnostic_figures(cfg, cache)
    runs = []
    for model, recipe in recipes.items():
        for filtering in ("P0", "P1"):
            run, _ = condition(
                cfg,
                cache,
                model,
                filtering,
                recipe["feature_set"],
                resume=resume,
                lr=recipe["lr"],
                hidden=recipe["hidden"],
            )
            runs.append(run)
            run_figures(cfg, cache, run)
            # Preserve validation evidence at the exact requested epoch milestones.
            for epoch in (60, 120):
                if (run / f"best_through_{epoch}.pt").exists():
                    evaluate_run(
                        cfg,
                        cache,
                        run,
                        checkpoint=f"best_through_{epoch}.pt",
                        label=run.name + f"_through_{epoch}",
                    )
    freeze_selection(cfg, cache, runs)
    if cfg["evaluation"]["final_test"]:
        final_evaluation(cfg, cache)
        for run in runs:
            run_figures(cfg, cache, run, "test")
    return {"runs": [run.name for run in runs], "test_requested": cfg["evaluation"]["final_test"]}

remember_source("study.txt")

### Synthetic fixture (only for software checks)

In [ ]:
"""Small synthetic WFDB fixtures for software checks, never research evidence."""

import copy
from pathlib import Path

import numpy as np
import pandas as pd
import wfdb



def synthetic_dataset(root, patients=(4, 2, 2), seed=42):
    root = Path(root)
    folder = root / "records100" / "00000"
    folder.mkdir(parents=True, exist_ok=True)
    rng, rows = np.random.default_rng(seed), []
    time = np.arange(1000) / 100
    index, patient = 1, 1
    for split, count in zip((1, 9, 10), patients, strict=True):
        for _ in range(count):
            for record_number in range(2):
                wave = np.column_stack(
                    [
                        0.1 * np.sin(2 * np.pi * (1 + patient / 50) * time + lead / 7)
                        + 0.01 * rng.normal(size=1000)
                        for lead in range(12)
                    ]
                )
                name = f"{index:05d}_lr"
                wfdb.wrsamp(
                    name,
                    fs=100,
                    units=["mV"] * 12,
                    sig_name=LEADS,
                    p_signal=wave,
                    fmt=["16"] * 12,
                    write_dir=str(folder),
                )
                rows.append(
                    {
                        "ecg_id": index,
                        "patient_id": patient,
                        "strat_fold": split,
                        "filename_lr": f"records100/00000/{name}",
                        "age": 300 if patient == 1 else 30 + patient,
                        "sex": patient % 2,
                        "recording_date": f"1990-01-{record_number + 1:02d} 12:00:00",
                        "scp_codes": "{'NORM': 100}",
                        "height": 170 + patient % 3,
                        "weight": 70 + patient % 4,
                    }
                )
                index += 1
            patient += 1
    pd.DataFrame(rows).to_csv(root / "ptbxl_database.csv", index=False)
    pd.DataFrame({"code": ["NORM"], "diagnostic": [1], "diagnostic_class": ["NORM"]}).set_index(
        "code"
    ).to_csv(root / "scp_statements.csv")
    sums = []
    for path in sorted(root.rglob("*")):
        if path.is_file() and path.name != "SHA256SUMS.txt":
            sums.append(f"{file_hash(path)}  {path.relative_to(root).as_posix()}")
    (root / "SHA256SUMS.txt").write_text("\n".join(sums) + "\n")
    return root


def fixture_config(work, base=None):
    work = Path(work)
    cfg = copy.deepcopy(base) if base else load_config()
    cfg["study_id"] = "synthetic_software_validation"
    cfg["data"]["root"] = str(synthetic_dataset(work / "synthetic_data"))
    cfg["data"]["synthetic"] = True
    cfg["paths"] = {"cache": str(work / "cache"), "results": str(work / "results")}
    cfg["cohorts"]["pilot"] = {"train": 4, "val": 2}
    cfg["cohorts"]["medium"] = {"train": 4, "val": None}
    cfg["model"].update(hidden=8, head_width=16, static_embedding=4, dropout=0.0)
    cfg["training"].update(
        epochs=2,
        pilot_epochs=2,
        full_epochs=2,
        batch_size=4,
        threads=1,
        allow_full_cpu=True,
        device="cpu",
        learning_rates=[0.0005],
    )
    cfg["evaluation"]["bootstrap_samples"] = 30
    return cfg

remember_source("fixture.txt")

## 3. Locate the real PTB-XL data

This cell uses the dataset folder resolved in the first check. The default is
the configured Drive folder in Colab and a local folder elsewhere. If you set
`DATA_SOURCE='physionet'`, this cell downloads the official 100 Hz dataset.

In [ ]:
cfg = load_config()
if VALIDATION_FIXTURE:
    cfg = fixture_config(Path(VALIDATION_FIXTURE), cfg)
    print('SYNTHETIC SOFTWARE FIXTURE: no PTB-XL performance claims.')
else:
    cfg['study_id'] = STUDY_ID
    cfg['run_mode'] = RUN_MODE
    cfg['training']['batch_size'] = BATCH_SIZE
    cfg['training']['pilot_epochs'] = PILOT_EPOCHS
    cfg['training']['epochs'] = 60 if RUN_MODE == 'full' else EPOCHS
    cfg['training']['full_epochs'] = 120
    cfg['training']['allow_full_cpu'] = ALLOW_FULL_CPU
    cfg['training']['tune_hidden_128'] = TUNE_HIDDEN_128
    cfg['training']['device'] = TRAINING_DEVICE
    cfg['features']['prior_history'] = ALLOW_PRIOR_HISTORY
    cfg['cohorts']['pilot'] = {'train': PILOT_TRAIN_PATIENTS, 'val': PILOT_VAL_PATIENTS}
    cfg['cohorts']['medium'] = {'train': MEDIUM_TRAIN_PATIENTS, 'val': MEDIUM_VAL_PATIENTS}
    if runtime == 'colab':
        cache_dir = COLAB_CACHE_DIR
        results_dir = DRIVE_OUTPUT if SAVE_TO_DRIVE else COLAB_RESULTS_DIR
    else:
        cache_dir = LOCAL_CACHE_DIR
        results_dir = LOCAL_RESULTS_DIR
    cfg['paths']['cache'] = str(Path(cache_dir).expanduser())
    cfg['paths']['results'] = str(Path(results_dir).expanduser())
    if data_source == 'physionet':
        download(dataset_root)
    cfg['data']['root'] = str(dataset_root)

cfg['evaluation']['final_test'] = False
validate(cfg)
seed_everything(cfg['training']['seed'], cfg['training']['threads'])
for required in (*required_dataset_files, 'records100'):
    if not (Path(cfg['data']['root']) / required).exists():
        raise FileNotFoundError(f"Missing dataset entry {required} under {cfg['data']['root']}.")
write_json(study_dir(cfg) / 'notebook_configuration.json', cfg)
print('Dataset:', Path(cfg['data']['root']).resolve())
print('Results:', study_dir(cfg).resolve())

## 4. Patient folds and leakage audit
`ecg_id` identifies a waveform; `patient_id` groups repeated ECGs; `strat_fold`
defines the official patient-safe partition; `filename_lr` locates the native
100-Hz WFDB pair. Timestamps may be used only for strictly earlier same-patient
annotation history, never current or equal-time labels. Preparation verifies
checksums, units, rate, shape, and finite values and preserves exact ID manifests. Counts below are for the development cohort (medium
screening when full mode is selected). Cohort settings count patients, not ECGs.
Full training uses all official training and validation patients; pilot/medium
limits never change the reserved test size.

In [ ]:
import copy
import pandas as pd

development_cfg = copy.deepcopy(cfg)
development_cfg['run_mode'] = 'pilot' if cfg['run_mode'] == 'pilot' else 'medium'
cache, manifest = prepare(development_cfg)
display(pd.DataFrame(manifest['splits']).T)
reserved = manifest['reserved_test']
print(f"Reserved fold 10: {reserved['patients']:,} patients, {reserved['records']:,} ECGs.")
display(audit(development_cfg, cache))
diagnostic_figures(development_cfg, cache)
print('Fold 10 remains reserved. No test metrics were calculated.')

## 5. Models, preprocessing, and tensor shapes
P0 and P1 use the same scaler fitted solely on raw training history and the same
unfiltered future target. P1 is causal; it can change phase, ST morphology, QRS
detail, and initial samples. It is not assumed to improve forecasting.
Both independent model classes use the shared training loop defined above.

In [ ]:
scaler_p0 = prepare_transforms(cache, development_cfg, 'P0')
scaler_p1 = prepare_transforms(cache, development_cfg, 'P1')
assert scaler_p0 == scaler_p1
for model_class in (LSTMForecaster, CNNLSTMForecaster):
    model = model_class(static_dim=4, horizon=cfg['data']['horizon'], **cfg['model'])
    with torch.inference_mode():
        prediction = model(torch.zeros(2, cfg['data']['history'], 12), torch.zeros(2, 4))
    print(model_class.__name__, 'output:', tuple(prediction.shape),
          'active parameters:', sum(p.numel() for p in model.parameters() if p.requires_grad))

## 6. Pilot or controlled screening
Pilot runs two epochs and reports software pass/fail, not research rankings.
Medium screening compares model×P0/P1 with ECG-only inputs and then compares F0/F1
using validation. The optional SINGLE_RUN controls run one explicitly chosen experiment.

In [ ]:
if SINGLE_RUN and not VALIDATION_FIXTURE:
    run = train_run(development_cfg, cache, ARCHITECTURE, FILTERING, FEATURE_SET, resume=RESUME)
    evaluate_run(development_cfg, cache, run)
elif cfg['run_mode'] == 'pilot':
    display(smoke(cfg, resume=RESUME))
else:
    display(screen(cfg, resume=RESUME))

## 7. Learning-rate and feature-selection experiments
For medium/full workflows, tune the three predefined learning rates on validation.
Run training-only mRMR and grouped validation permutation as a separate expanded-feature
extension, keeping all twelve waveform leads. Prior diagnoses stay disabled unless
their chronology is independently audited. ARIMA remains deferred.

In [ ]:
if cfg['run_mode'] in ('medium', 'full') and not SINGLE_RUN:
    display(tune(cfg, resume=RESUME))
    display(feature_extension(cfg, resume=RESUME))
else:
    print('Longer screening/tuning and feature-selection runs are pending; pilot mode does not fabricate them.')

## 8. FULL RUN — four matched 120-epoch conditions
This cell runs only when RUN_MODE='full'. The selected feature set and hyperparameters
are fixed within each architecture's P0/P1 pair. All official training/validation
patients are used. Epoch 60 and 120 evidence, best checkpoints, optimizer/RNG state,
and logs are saved; no early stopping shortens the requested budget.

In [ ]:
if cfg['run_mode'] == 'full' and not SINGLE_RUN:
    display(full(cfg, resume=RESUME))
else:
    print('Full-population 120-epoch training is pending, not a completed result.')

## 9. Locked final evaluation
FINAL_TEST must be explicitly enabled. The four validation-selected checkpoint hashes,
raw scaler, configurations and cohort are frozen first. Complete fold 10 is evaluated
with a ledger: completed conditions are checked and reused; interrupted conditions
resume without changing selections. Baselines use the identical raw 7s→3s target.

In [ ]:
if cfg['run_mode'] == 'full' and FINAL_TEST and not SINGLE_RUN:
    cfg['evaluation']['final_test'] = True
    final_cache, _ = prepare(cfg)
    display(final_evaluation(cfg, final_cache))
    for name in verify_lock(cfg, final_cache)['runs']:
        run_figures(cfg, final_cache, study_dir(cfg) / 'full/checkpoints' / name, 'test')
else:
    print('Final test evaluation pending or intentionally disabled. Validation remains the selection source.')

## 10. Automatic evidence summary and export
The final sections appear in the required order. Missing runs remain pending.
Every numerical result is tied to its saved cohort, scaler, configuration and checkpoint.
Figures explain what is visible and what conclusion is supported; example selection
and single-run, cohort, sampling-rate, and morphology limitations remain explicit.

In [ ]:
from IPython.display import Markdown, Image, display

report_path = generate(cfg)
summary = read_json(report_path.parent / 'summary.json')
if summary['warning']:
    display(Markdown(summary['warning']))
figures = report_path.parent / 'figures'
medium_figures = study_dir(cfg) / 'medium' / 'figures' / 'comparisons'
image_groups = {
    'Patient-safe folds': list(figures.glob('data/fold_counts.png')),
    'Preprocessing': list(figures.glob('data/filtered_history.png')) + list(figures.glob('data/psd.png')),
    'Training': list(figures.glob('*/learning_curves.png'))[:2],
    'Twelve-lead prediction': list(figures.glob('*/*pre_specified_12_leads.png'))[:2] + list(figures.glob('*/*median_error_12_leads.png'))[:2],
    'Evidence-based comparison': list(figures.glob('comparisons/*.png'))[:3] + (
        [p for p in medium_figures.glob('*.png') if any(k in p.name for k in ('learning_rate', 'feature_sets'))]
        if cfg['run_mode'] == 'full' else []
    ),
}
for section in SUMMARY_ORDER:
    display(Markdown('## ' + section + '\n\n' + summary['sections'][section]))
    for path in image_groups.get(section, []):
        display(Image(filename=str(path)))
        caption = path.with_suffix('.md')
        if caption.exists():
            display(Markdown(caption.read_text(encoding='utf-8')))
archive_path = bundle(cfg)
print('Saved report:', report_path)
print('Saved ZIP:', archive_path.resolve())